# Module 03: Advanced SQL for Data Engineering

**Program:** Quintrix Jr. Data Engineer Training  
**Duration:** 5 hours  
**Instructor:** Sunil Mogadati

---

**Today's Agenda:**

| # | Topic | Time |
|---|-------|------|
| 1 | Database Setup | 10 min |
| 2 | Advanced Aggregations & Pivoting | 40 min |
| 3 | Subqueries — Queries Inside Queries | 20 min |
| 4 | Common Table Expressions (CTEs) | 25 min |
| 5 | Window Functions — The DE Power Tool | 45 min |
| 6 | Date/Time Mastery for Pipelines | 20 min |
| 7 | Data Transformation Patterns | 25 min |
| 8 | Query Optimization & Execution Plans | 40 min |
| 9 | Incremental Loading Strategies | 25 min |
| 10 | SQL Challenge Lab | 30 min |
| 11 | Key Takeaways | 5 min |
| 12 | Homework & Preview | 15 min |

---

## 1. Database Setup

We rebuild the same call center database from M02. Run this cell once — every exercise in this module uses these tables.

**Schema reminder:** `calls` → `orders` → `payments`, with `sources` and `clients` as reference tables. The platform handles both live agent and VA (virtual agent) calls.

| Table | Rows | Key Columns |
|-------|------|-------------|
| `calls` | 40 | call_id, dnis, call_started_at, call_ended_at, disposition, channel |
| `orders` | 15 | order_id, call_id (FK), sku, total |
| `payments` | 15 | transaction_id, order_id (FK), amount, status |
| `sources` | 6 | dnis (PK), campaign_name, client_name, channel |
| `clients` | 3 | client_id, client_name, industry |

> **(PK = Primary Key: unique row ID. FK = Foreign Key: link to another table.)**


In [ ]:
# ── HELLO WORLD: SQL in 4 lines ───────────────────────────────────────────
# Before we build the full database, let's see what SQL actually IS.
# This cell creates a tiny database, adds 3 rows, and reads them back.
# That's ALL SQL is: store data, ask questions about it.

import sqlite3

# Connect to an in-memory database (lives only while Python is running)
hw_conn = sqlite3.connect(":memory:")
hw_cursor = hw_conn.cursor()

# CREATE TABLE: define the shape of your data (columns + data types)
hw_cursor.execute(
    "CREATE TABLE agents ("
    "  name    TEXT,"       # text string column
    "  calls   INTEGER,"    # whole number column  
    "  channel TEXT"        # 'live_agent' or 'va'
    ")"
)

# INSERT: add rows of data into the table
hw_cursor.execute("INSERT INTO agents VALUES ('Maria',  42, 'live_agent')")
hw_cursor.execute("INSERT INTO agents VALUES ('Derek',  38, 'live_agent')")
hw_cursor.execute("INSERT INTO agents VALUES ('Aria',   91, 'va')")

# SELECT *: read ALL columns and ALL rows back out
# The * means "give me everything"
rows = hw_cursor.execute("SELECT * FROM agents").fetchall()

print("=== All Agents ===")
for row in rows:
    # row is a tuple: (name, calls, channel)
    print(f"  {row[0]:<8} handled {row[1]} calls via {row[2]}")

# WHERE: filter — only show the rows you care about
print("\n=== Live Agents Only (WHERE channel = 'live_agent') ===")
live = hw_cursor.execute(
    "SELECT * FROM agents WHERE channel = 'live_agent'"
).fetchall()
for row in live:
    print(f"  {row[0]}: {row[1]} calls")

# ORDER BY + LIMIT: sort and take the top result
print("\n=== Top Agent by Calls ===")
top = hw_cursor.execute(
    "SELECT name, calls FROM agents ORDER BY calls DESC LIMIT 1"
).fetchone()
print(f"  {top[0]} with {top[1]} calls")

print("\n--- That's all SQL is: store data, ask questions. ---")
print("Now we'll build the full call center database for real exercises.")


In [ ]:
# ── SECTION 1: Import and connect ─────────────────────────────────────────
# sqlite3 is Python's built-in SQL engine — no install needed.
# In production you'd connect to BigQuery, Postgres, or Snowflake instead.
import sqlite3

# ":memory:" creates the database in RAM, not on disk.
# WHY: In-memory DBs are perfect for learning — they reset every run,
# so you can't accidentally corrupt a real file.
conn = sqlite3.connect(":memory:")

# cursor is the object you send SQL commands through.
# Think of it as the "pen" that writes queries to the database.
cursor = conn.cursor()

# ── SECTION 2: Create the calls table ─────────────────────────────────────
# calls is the FACT TABLE — every call that came in gets one row here.
# A fact table records events. Dimension tables (sources, clients) describe them.
cursor.execute("""
CREATE TABLE calls (
    call_id         TEXT PRIMARY KEY,  -- Unique ID for each call (e.g. 'call_001')
    dnis            TEXT NOT NULL,     -- DNIS = Dialed Number Identification Service
                                       -- The phone number the customer dialed (e.g. 800-555-1234)
                                       -- NOT NULL = this column must always have a value
    caller_ani      TEXT,              -- ANI = Automatic Number Identification (caller's phone number)
                                       -- No NOT NULL here — callers can block their number
    call_started_at TEXT NOT NULL,     -- ISO 8601 timestamp stored as text (SQLite has no DATETIME type)
                                       -- Format: '2026-03-10T13:00:00Z' — the Z means UTC timezone
    call_ended_at   TEXT,              -- NULL if the call is still active or was never completed
    disposition     TEXT,              -- Outcome: 'completed', 'dropped', or 'transferred'
    sentiment       TEXT,              -- AI-scored sentiment: 'positive', 'neutral', 'negative'
    channel         TEXT NOT NULL      -- 'live_agent' = human answered | 'va' = virtual agent (bot)
)
""")

# ── SECTION 3: Create the orders table ────────────────────────────────────
# orders is also a fact table — one row per purchase made during a call.
# Not every call results in an order (most don't), so call_id appears in
# both tables — that's the FOREIGN KEY relationship.
cursor.execute("""
CREATE TABLE orders (
    order_id    TEXT PRIMARY KEY,
    call_id     TEXT,          -- FK = Foreign Key: links back to the calls table
                               -- WHY: This lets us JOIN orders to calls to see
                               -- which orders came from which calls
    sku         TEXT NOT NULL, -- SKU = Stock Keeping Unit (product identifier)
    subtotal    REAL,          -- REAL = floating point number (decimal)
    tax         REAL,
    shipping    REAL,
    total       REAL,          -- subtotal + tax + shipping
    FOREIGN KEY (call_id) REFERENCES calls(call_id)
    -- ↑ This tells SQLite: call_id here must match a call_id in the calls table
)
""")

# ── SECTION 4: Create the payments table ──────────────────────────────────
# payments records the financial transaction for each order.
# One order → one payment attempt. Status tells us if it worked.
cursor.execute("""
CREATE TABLE payments (
    transaction_id  TEXT PRIMARY KEY,
    order_id        TEXT,      -- FK links to orders table
    auth_code       TEXT,      -- Authorization code from the payment processor
                               -- NULL if the payment was declined (no auth issued)
    amount          REAL,
    status          TEXT,      -- 'approved' or 'declined'
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
)
""")

# ── SECTION 5: Create dimension tables ────────────────────────────────────
# DIMENSION TABLES store descriptive attributes about things.
# They don't record events — they describe who/what/where.

# sources maps each DNIS phone number to a campaign and client.
# WHY: Calls only store the dnis number. To know the campaign name,
# you JOIN calls to sources on dnis.
cursor.execute("""
CREATE TABLE sources (
    dnis            TEXT PRIMARY KEY,   -- The phone number (also PK — each DNIS is unique)
    campaign_name   TEXT NOT NULL,
    client_name     TEXT NOT NULL,
    channel         TEXT NOT NULL       -- 'live_agent' or 'va' — matches calls.channel
)
""")

# clients stores high-level client info.
# In this schema: clients → sources → calls → orders → payments
# That's the full chain from client down to payment.
cursor.execute("""
CREATE TABLE clients (
    client_id   INTEGER PRIMARY KEY,   -- INTEGER PK auto-increments in SQLite
    client_name TEXT NOT NULL,
    industry    TEXT
)
""")

# ── SECTION 6: Insert reference data (clients and sources) ─────────────────
# We insert dimension data first because calls.dnis references sources.dnis
# (foreign key constraint — the parent must exist before the child)
clients_data = [
    (1, "Acme Corp", "Retail"),
    (2, "Pinnacle Health", "Healthcare"),
    (3, "Summit Financial", "Financial Services"),
]
# executemany() inserts multiple rows at once — more efficient than looping
cursor.executemany("INSERT INTO clients VALUES (?, ?, ?)", clients_data)
# ↑ ? = parameterized placeholder. SQLite fills in the values safely.
# This prevents SQL injection — never use f-strings to build SQL with user data.

sources_data = [
    # (dnis,           campaign_name,              client_name,         channel)
    ("8005551234", "Acme Spring Sale",          "Acme Corp",         "live_agent"),
    ("8005551235", "Acme Rewards Program",      "Acme Corp",         "va"),
    ("8005552345", "Pinnacle Wellness Plan",    "Pinnacle Health",   "live_agent"),
    ("8005552346", "Pinnacle Rx Refill",        "Pinnacle Health",   "va"),
    ("8005553456", "Summit Auto Loan",          "Summit Financial",  "live_agent"),
    ("8005553457", "Summit Credit Card",        "Summit Financial",  "va"),
]
cursor.executemany("INSERT INTO sources VALUES (?, ?, ?, ?)", sources_data)

# ── SECTION 7: Insert calls data (40 rows) ────────────────────────────────
# 40 calls across 2 days (March 10-11, 2026), 6 different DNIS numbers.
# Intentional data patterns: some calls drop, some transfer, most complete.
# This mirrors a real call center — not every call is a success.
calls_data = [
    # (call_id, dnis, caller_ani, call_started_at, call_ended_at, disposition, sentiment, channel)
    ("call_001", "8005551234", "3135559876", "2026-03-10T13:00:00Z", "2026-03-10T13:07:45Z", "completed", "positive", "live_agent"),
    ("call_002", "8005551234", "2485555678", "2026-03-10T13:30:00Z", "2026-03-10T13:32:15Z", "dropped", "negative", "live_agent"),
    ("call_003", "8005552345", "7345551234", "2026-03-10T14:00:00Z", "2026-03-10T14:12:30Z", "completed", "neutral", "live_agent"),
    ("call_004", "8005552346", "3135554321", "2026-03-10T14:30:00Z", "2026-03-10T14:35:00Z", "transferred", "neutral", "va"),
    ("call_005", "8005553456", "2485559999", "2026-03-10T15:00:00Z", "2026-03-10T15:08:20Z", "completed", "positive", "live_agent"),
    ("call_006", "8005551235", "5865551111", "2026-03-10T15:30:00Z", "2026-03-10T15:31:10Z", "dropped", "negative", "va"),
    ("call_007", "8005553457", "3135558888", "2026-03-10T16:00:00Z", "2026-03-10T16:06:45Z", "completed", "positive", "va"),
    ("call_008", "8005551234", "7345557777", "2026-03-10T16:30:00Z", "2026-03-10T16:42:00Z", "completed", "neutral", "live_agent"),
    ("call_009", "8005552345", "2485556666", "2026-03-10T17:00:00Z", "2026-03-10T17:01:30Z", "dropped", "negative", "live_agent"),
    ("call_010", "8005553456", "3135553333", "2026-03-10T17:30:00Z", "2026-03-10T17:38:15Z", "completed", "positive", "live_agent"),
    ("call_011", "8005551234", "5865552222", "2026-03-10T18:00:00Z", "2026-03-10T18:09:30Z", "completed", "neutral", "live_agent"),
    ("call_012", "8005552346", "7345554444", "2026-03-10T18:30:00Z", "2026-03-10T18:33:00Z", "transferred", "neutral", "va"),
    ("call_013", "8005551235", "3135551111", "2026-03-10T19:00:00Z", "2026-03-10T19:05:45Z", "completed", "positive", "va"),
    ("call_014", "8005553457", "2485553333", "2026-03-10T19:30:00Z", "2026-03-10T19:31:00Z", "dropped", "negative", "va"),
    ("call_015", "8005551234", "5865554444", "2026-03-10T20:00:00Z", "2026-03-10T20:11:20Z", "completed", "positive", "live_agent"),
    ("call_016", "8005552345", "7345558888", "2026-03-10T20:30:00Z", "2026-03-10T20:36:00Z", "completed", "neutral", "live_agent"),
    ("call_017", "8005553456", "3135559999", "2026-03-10T21:00:00Z", "2026-03-10T21:02:00Z", "dropped", "negative", "live_agent"),
    ("call_018", "8005551235", "2485551111", "2026-03-10T21:30:00Z", "2026-03-10T21:40:15Z", "completed", "neutral", "va"),
    ("call_019", "8005552346", "5865557777", "2026-03-10T22:00:00Z", "2026-03-10T22:04:30Z", "transferred", "neutral", "va"),
    ("call_020", "8005553457", "7345552222", "2026-03-10T22:30:00Z", "2026-03-10T22:37:45Z", "completed", "positive", "va"),
    ("call_021", "8005551234", "3135556666", "2026-03-11T00:00:00Z", "2026-03-11T00:09:30Z", "completed", "neutral", "live_agent"),
    ("call_022", "8005552345", "2485558888", "2026-03-11T00:30:00Z", "2026-03-11T00:31:45Z", "dropped", "negative", "live_agent"),
    ("call_023", "8005553456", "5865553333", "2026-03-11T01:00:00Z", "2026-03-11T01:06:00Z", "completed", "positive", "live_agent"),
    ("call_024", "8005551235", "7345559999", "2026-03-11T01:30:00Z", "2026-03-11T01:38:20Z", "completed", "positive", "va"),
    ("call_025", "8005553457", "3135552222", "2026-03-11T02:00:00Z", "2026-03-11T02:03:15Z", "transferred", "neutral", "va"),
    ("call_026", "8005551234", "2485554444", "2026-03-11T13:00:00Z", "2026-03-11T13:05:30Z", "completed", "positive", "live_agent"),
    ("call_027", "8005552345", "5865556666", "2026-03-11T13:30:00Z", "2026-03-11T13:32:00Z", "dropped", "negative", "live_agent"),
    ("call_028", "8005553456", "7345553333", "2026-03-11T14:00:00Z", "2026-03-11T14:10:45Z", "completed", "neutral", "live_agent"),
    ("call_029", "8005551235", "3135557777", "2026-03-11T14:30:00Z", "2026-03-11T14:34:00Z", "transferred", "neutral", "va"),
    ("call_030", "8005552346", "2485552222", "2026-03-11T15:00:00Z", "2026-03-11T15:08:30Z", "completed", "positive", "va"),
    ("call_031", "8005553457", "5865559999", "2026-03-11T15:30:00Z", "2026-03-11T15:31:20Z", "dropped", "negative", "va"),
    ("call_032", "8005551234", "7345556666", "2026-03-11T16:00:00Z", "2026-03-11T16:07:00Z", "completed", "neutral", "live_agent"),
    ("call_033", "8005552345", "3135554444", "2026-03-11T16:30:00Z", "2026-03-11T16:41:30Z", "completed", "positive", "live_agent"),
    ("call_034", "8005553456", "2485557777", "2026-03-11T17:00:00Z", "2026-03-11T17:01:45Z", "dropped", "negative", "live_agent"),
    ("call_035", "8005551235", "5865558888", "2026-03-11T17:30:00Z", "2026-03-11T17:36:15Z", "completed", "positive", "va"),
    ("call_036", "8005552346", "7345551111", "2026-03-11T18:00:00Z", "2026-03-11T18:03:30Z", "transferred", "neutral", "va"),
    ("call_037", "8005553457", "3135558888", "2026-03-11T18:30:00Z", "2026-03-11T18:39:45Z", "completed", "neutral", "va"),
    ("call_038", "8005551234", "2485559876", "2026-03-11T19:00:00Z", "2026-03-11T19:06:20Z", "completed", "positive", "live_agent"),
    ("call_039", "8005552345", "5865554321", "2026-03-11T19:30:00Z", "2026-03-11T19:32:00Z", "dropped", "negative", "live_agent"),
    ("call_040", "8005553456", "7345559876", "2026-03-11T20:00:00Z", "2026-03-11T20:07:30Z", "completed", "positive", "live_agent"),
]
cursor.executemany("INSERT INTO calls VALUES (?, ?, ?, ?, ?, ?, ?, ?)", calls_data)

# ── SECTION 8: Insert orders and payments ─────────────────────────────────
# Only 15 of 40 calls result in orders — realistic ~37% conversion rate.
# Notice: each SKU prefix maps to a client (AC=Acme, PH=Pinnacle, SF=Summit)
orders_data = [
    # (order_id, call_id, sku, subtotal, tax, shipping, total)
    ("ord_001", "call_001", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_002", "call_003", "SKU-PH-2001", 99.99, 8.00, 11.99, 119.98),
    ("ord_003", "call_005", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_004", "call_008", "SKU-AC-1002", 139.97, 11.20, 8.80, 159.97),
    ("ord_005", "call_010", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_006", "call_011", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_007", "call_015", "SKU-AC-1003", 89.99, 7.20, 5.80, 102.99),
    ("ord_008", "call_016", "SKU-PH-2002", 59.95, 4.80, 5.20, 69.95),
    ("ord_009", "call_020", "SKU-SF-3002", 149.99, 12.00, 7.00, 168.99),
    ("ord_010", "call_021", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_011", "call_023", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_012", "call_026", "SKU-AC-1002", 139.97, 11.20, 8.80, 159.97),
    ("ord_013", "call_030", "SKU-PH-2001", 99.99, 8.00, 11.99, 119.98),
    ("ord_014", "call_033", "SKU-PH-2002", 59.95, 4.80, 5.20, 69.95),
    ("ord_015", "call_040", "SKU-SF-3002", 149.99, 12.00, 7.00, 168.99),
]
cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)", orders_data)

# 2 of 15 payments are declined — intentional for data quality exercises later.
# auth_code is NULL when declined (no authorization was issued by the bank).
payments_data = [
    # (transaction_id, order_id, auth_code, amount, status)
    ("txn_001", "ord_001", "AUTH-7821", 79.99, "approved"),
    ("txn_002", "ord_002", "AUTH-7822", 119.98, "approved"),
    ("txn_003", "ord_003", "AUTH-7823", 49.95, "approved"),
    ("txn_004", "ord_004", "AUTH-7824", 159.97, "approved"),
    ("txn_005", "ord_005", "AUTH-7825", 49.95, "approved"),
    ("txn_006", "ord_006", None, 79.99, "declined"),        # ← auth_code is NULL (declined)
    ("txn_007", "ord_007", "AUTH-7827", 102.99, "approved"),
    ("txn_008", "ord_008", "AUTH-7828", 69.95, "approved"),
    ("txn_009", "ord_009", "AUTH-7829", 168.99, "approved"),
    ("txn_010", "ord_010", "AUTH-7830", 79.99, "approved"),
    ("txn_011", "ord_011", "AUTH-7831", 49.95, "approved"),
    ("txn_012", "ord_012", None, 159.97, "declined"),       # ← auth_code is NULL (declined)
    ("txn_013", "ord_013", "AUTH-7833", 119.98, "approved"),
    ("txn_014", "ord_014", "AUTH-7834", 69.95, "approved"),
    ("txn_015", "ord_015", "AUTH-7835", 168.99, "approved"),
]
cursor.executemany("INSERT INTO payments VALUES (?, ?, ?, ?, ?)", payments_data)

# commit() persists all inserts to the in-memory database.
# Without commit(), the data exists only in the transaction buffer.
conn.commit()

# ── SECTION 9: Verify the setup ────────────────────────────────────────────
# Quick sanity check — confirm row counts match the schema table above.
print("=== Database Ready ===")
for t in ['calls', 'orders', 'payments', 'sources', 'clients']:
    n = cursor.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")
# You should see: calls=40, orders=15, payments=15, sources=6, clients=3


---

## 2. Advanced Aggregations & Pivoting

**One-line answer:** GROUP BY gives you totals. HAVING filters those totals. CASE WHEN pivots them into columns.

In M02, you used `GROUP BY`, `COUNT`, and `SUM`. Now we go deeper.

### HAVING vs. WHERE

| Clause | Filters... | When It Runs |
|--------|-----------|-------------|
| `WHERE` | Individual rows BEFORE grouping | Before GROUP BY |
| `HAVING` | Groups AFTER aggregation | After GROUP BY |

```sql
-- WHERE: filter rows, then group
SELECT dnis, COUNT(*) FROM calls WHERE channel = 'va' GROUP BY dnis

-- HAVING: group all rows, then filter groups
SELECT dnis, COUNT(*) FROM calls GROUP BY dnis HAVING COUNT(*) > 5
```

> **Rule:** If you're filtering on a column value, use `WHERE`. If you're filtering on an aggregate (COUNT, SUM, AVG), use `HAVING`.

> **Analogy:** Think of WHERE as security at the door — it checks who gets in before the party starts. HAVING is the bouncer inside the VIP section — it checks conditions after the groups have already formed.


In [ ]:
# WHY HAVING instead of WHERE here?
# HAVING filters AFTER the database has already grouped and counted the rows.
# WHERE runs BEFORE grouping — you can't use COUNT(*) inside WHERE because
# the count doesn't exist yet at that point in the execution order.
# BEGINNER NOTE: SQL execution order is: FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY
# It's NOT top-to-bottom like Python. HAVING runs after GROUP BY.

# HAVING: Find campaigns with more than 5 calls
print("=== Campaigns with More Than 5 Calls (HAVING) ===")
print(f"{'Campaign':<26} {'Client':<20} {'Calls':>6}")
print("-" * 54)

results = cursor.execute("""
    SELECT s.campaign_name, s.client_name, COUNT(*) as call_count
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    -- WHY JOIN? dnis is just a phone number in 'calls'.
    -- We JOIN to 'sources' to get the human-readable campaign name.
    GROUP BY s.campaign_name, s.client_name
    HAVING COUNT(*) > 5   -- Filter groups (not rows) — only campaigns with 6+ calls
    ORDER BY call_count DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:<20} {row[2]:>6}")

# Compare: WHERE + HAVING together
# This shows both clauses in one query — WHERE fires first, HAVING fires last
print("\n=== Completed Calls Per Campaign, Only Campaigns with 3+ Completions ===")
print(f"{'Campaign':<26} {'Completed':>10}")
print("-" * 38)

results = cursor.execute("""
    SELECT s.campaign_name, COUNT(*) as completed
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    WHERE c.disposition = 'completed'          -- WHERE filters rows first (only completed calls enter GROUP BY)
    GROUP BY s.campaign_name
    HAVING COUNT(*) >= 3                        -- HAVING filters groups after (only campaigns with 3+ completions)
    ORDER BY completed DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:>10}")

# You should see: Acme Spring Sale at top with 5+ completions
print("\nWHERE ran first (kept only completed), then GROUP BY, then HAVING filtered groups.")


### Pivoting with CASE WHEN

SQL doesn't have a built-in PIVOT command in most engines. Instead, you use `CASE WHEN` inside aggregations to turn row values into columns.

```sql
-- Turn disposition values into columns
SELECT campaign_name,
       SUM(CASE WHEN disposition = 'completed' THEN 1 ELSE 0 END) as completed,
       SUM(CASE WHEN disposition = 'dropped' THEN 1 ELSE 0 END) as dropped
FROM calls
GROUP BY campaign_name
```

> **Where you'll use this:** BigQuery, PySpark SQL, and reporting queries all use CASE WHEN pivots. dbt (a SQL transformation tool for production data warehouses) models use them constantly.

In [ ]:
# WHY: CASE WHEN inside SUM() is the standard SQL pivot technique.
# Each CASE WHEN acts as a conditional counter:
#   SUM(CASE WHEN disposition = 'completed' THEN 1 ELSE 0 END)
#   reads as: "for each row, add 1 if completed, 0 otherwise, then sum it all up"
# This turns row-level values (one row per call) into column-level totals.

# CASE WHEN Pivot: Disposition breakdown by campaign
print("=== Disposition Breakdown by Campaign ===")
print(f"{'Campaign':<26} {'Total':>6} {'Done':>6} {'Drop':>6} {'Xfer':>6} {'Done%':>6}")
print("-" * 58)

results = cursor.execute("""
    SELECT s.campaign_name,
           COUNT(*) as total,
           -- Each CASE WHEN column counts one disposition type
           SUM(CASE WHEN c.disposition = 'completed'   THEN 1 ELSE 0 END) as completed,
           SUM(CASE WHEN c.disposition = 'dropped'     THEN 1 ELSE 0 END) as dropped,
           SUM(CASE WHEN c.disposition = 'transferred' THEN 1 ELSE 0 END) as transferred,
           -- WHY 1.0 instead of 1? Forces float division. THEN 1 / COUNT(*) would be integer division = 0.
           ROUND(SUM(CASE WHEN c.disposition = 'completed' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) as comp_pct
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    GROUP BY s.campaign_name
    ORDER BY total DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:>6} {row[2]:>6} {row[3]:>6} {row[4]:>6} {row[5]:>5}%")
# You should see 6 rows, one per campaign — totals across all dispositions

# Conversion rate by channel: LEFT JOIN keeps calls that have NO order
# WHY LEFT JOIN and not JOIN? A regular (INNER) JOIN would drop calls with no order.
# We want ALL calls to appear, with NULL for orders where none exist.
print("\n=== Conversion Rate by Channel ===")
print(f"{'Channel':<12} {'Calls':>6} {'Orders':>7} {'Conv%':>7} {'Revenue':>10}")
print("-" * 44)

results = cursor.execute("""
    SELECT c.channel,
           COUNT(DISTINCT c.call_id) as total_calls,  -- DISTINCT avoids double-counting if a call joins to multiple rows
           COUNT(DISTINCT o.order_id) as orders,       -- NULL order_ids from LEFT JOIN are skipped by COUNT
           ROUND(COUNT(DISTINCT o.order_id) * 100.0 / COUNT(DISTINCT c.call_id), 1) as conv_pct,
           -- COALESCE(SUM(...), 0): if SUM returns NULL (no orders), use 0 instead
           ROUND(COALESCE(SUM(o.total), 0), 2) as revenue
    FROM calls c
    LEFT JOIN orders o ON c.call_id = o.call_id  -- LEFT JOIN: keep all calls even without an order
    GROUP BY c.channel
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>7} {row[3]:>6}% ${row[4]:>9.2f}")

print("\nCASE WHEN turns row values into columns — essential for cross-tab reports.")


---

## 3. Subqueries — Queries Inside Queries

**One-line answer:** A subquery is a SELECT inside another SELECT — it lets you use one query's result as input to another.

> **Analogy:** A scalar subquery is like asking a colleague a question mid-sentence: "Send me everyone who lasted longer than the average call" — the average is computed on the spot, right inside your request.

### Three Types of Subqueries

| Type | Where It Goes | Returns | Example Use |
|------|-------------|---------|-------------|
| **Scalar** | SELECT or WHERE | Single value | Compare to average |
| **Table** | FROM or IN | Set of rows/values | Filter against a list |
| **Correlated** | WHERE | Varies | Row-by-row comparison |

> **Performance note:** Correlated subqueries run once per row in the outer query — they can be slow on large tables. CTEs (Section 4) are often a cleaner alternative.

In [ ]:
# julianday() is SQLite's way to do datetime arithmetic.
# It converts a timestamp to a decimal day number (days since Nov 24, 4714 BC).
# Subtract two julianday values = elapsed days as a decimal.
# Multiply by 1440 (minutes per day) to get duration in minutes.
# Multiply by 86400 (seconds per day) to get duration in seconds.
# BigQuery equivalent: TIMESTAMP_DIFF(end, start, SECOND) — same idea, cleaner syntax.

# Scalar subquery: Calls longer than average duration
# WHY scalar subquery? We need the average to compare against each row.
# The inner SELECT returns ONE number. The outer WHERE uses that number as a threshold.
avg_dur = cursor.execute("""
    SELECT ROUND(AVG((julianday(call_ended_at) - julianday(call_started_at)) * 1440), 1)
    FROM calls
""").fetchone()[0]
print(f"Average call duration: {avg_dur} minutes\n")

print("=== Calls Longer Than Average Duration ===")
print(f"{'Call ID':<12} {'Channel':<12} {'Disposition':<14} {'Duration':>8}")
print("-" * 48)

results = cursor.execute("""
    SELECT call_id, channel, disposition,
           ROUND((julianday(call_ended_at) - julianday(call_started_at)) * 1440, 1) as dur_min
    FROM calls
    WHERE (julianday(call_ended_at) - julianday(call_started_at)) * 1440 > (
        -- This inner SELECT runs ONCE, returns the average, then the outer WHERE uses it
        SELECT AVG((julianday(call_ended_at) - julianday(call_started_at)) * 1440)
        FROM calls
    )
    ORDER BY dur_min DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<14} {row[3]:>7.1f}m")
# You should see roughly half the calls — those above the average duration
print(f"\n{len(results)} calls above average duration of {avg_dur} min")

# IN subquery: checks if a value is in a set returned by a SELECT.
# WHY IN over JOIN? Cleaner when you only need to filter (not retrieve columns from the other table).
# DNIS = Dialed Number Identification Service — the 800 number the customer called.
print("\n=== Campaigns With Dropped Calls (IN subquery) ===")
results = cursor.execute("""
    SELECT dnis, campaign_name, channel
    FROM sources
    WHERE dnis IN (
        -- Inner query: find all DNIS numbers that have at least one dropped call
        -- DISTINCT prevents duplicates (same DNIS could have many dropped calls)
        SELECT DISTINCT dnis FROM calls WHERE disposition = 'dropped'
    )
    -- Outer query: return the campaign info for those DNIS numbers
""").fetchall()

for row in results:
    print(f"  {row[1]} ({row[2]}) — DNIS {row[0]}")
# You should see all 6 campaigns — every DNIS had at least one drop in our dataset


In [ ]:
# EXISTS checks whether a subquery returns ANY rows — it doesn't care what those rows contain.
# WHY use EXISTS over IN? EXISTS stops as soon as it finds the first match (faster on large tables).
# The convention "SELECT 1" signals to the reader: "I don't care about the value, just existence."

# EXISTS: Completed calls that resulted in approved payments
print("=== Calls With Approved Payments (EXISTS) ===")
print(f"{'Call ID':<12} {'Channel':<12} {'Disposition':<14}")
print("-" * 38)

results = cursor.execute("""
    SELECT c.call_id, c.channel, c.disposition
    FROM calls c
    WHERE EXISTS (
        -- SELECT 1 is convention: EXISTS only checks if any row exists, not what it contains
        -- This subquery is CORRELATED — it references c.call_id from the outer query.
        -- It runs once per row in 'calls', checking if that specific call has an approved payment.
        SELECT 1
        FROM orders o
        JOIN payments p ON o.order_id = p.order_id
        WHERE o.call_id = c.call_id    -- ← correlation: ties inner query to current outer row
          AND p.status = 'approved'
    )
    ORDER BY c.call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<14}")
print(f"\n{len(results)} calls with approved payments")
# You should see 13 calls — 15 orders minus 2 declined payments

# NOT EXISTS: the logical opposite — find calls where NO matching row exists in the subquery.
# Use case: finding gaps. Completed calls that never generated revenue = missed opportunity.
print("\n=== Completed Calls With NO Order (NOT EXISTS) ===")
results = cursor.execute("""
    SELECT c.call_id, c.channel
    FROM calls c
    WHERE c.disposition = 'completed'    -- Only look at completed calls
      AND NOT EXISTS (                    -- ...that have NO matching order
          -- If this subquery returns zero rows, NOT EXISTS = TRUE and the outer row is kept
          SELECT 1 FROM orders o WHERE o.call_id = c.call_id
      )
    ORDER BY c.call_started_at
""").fetchall()

for row in results:
    print(f"  {row[0]} ({row[1]})")
print(f"\n{len(results)} completed calls had no order — potential missed conversions.")
# BEGINNER NOTE: NOT EXISTS is how you find orphans, gaps, and missing relationships.
# This same pattern in BigQuery/PySpark finds records that didn't make it through the pipeline.


---

## 4. Common Table Expressions (CTEs)

**One-line answer:** A CTE is a named temporary result set — it makes complex queries readable by breaking them into logical steps.

> **Analogy:** Think of a CTE like naming a calculation on a whiteboard before using it. Instead of nesting one formula inside another, you write each step up top with a label, then reference the labels. It reads top-to-bottom, like a recipe.

```sql
WITH step_one AS (
    SELECT ...          -- Calculate something
),
step_two AS (
    SELECT ...          -- Use step_one's result
    FROM step_one
)
SELECT * FROM step_two  -- Final output
```

### CTE vs. Subquery

| Aspect | Subquery | CTE |
|--------|---------|-----|
| Readability | Nested, hard to follow | Named steps, reads top-to-bottom |
| Reuse | Must repeat the subquery | Reference the CTE name multiple times |
| Debugging | Can't run inner query alone easily | Can comment out final SELECT, run CTE alone |
| Performance | Same as CTE in most engines | Same as subquery in most engines |

> **In practice:** Data engineers almost always prefer CTEs over subqueries. dbt models are entirely built on CTEs. BigQuery, Spark SQL, and every modern engine supports them.

In [ ]:
# WHY use a CTE here instead of just filtering inline?
# Without CTE: WHERE (julianday(call_ended_at) - julianday(call_started_at)) * 1440 > 8
# Problem: you'd have to write that long formula twice — once in SELECT, once in WHERE.
# With CTE: calculate duration ONCE in the CTE, then reference duration_min everywhere.
# The CTE also makes it easy to debug: comment out the final SELECT, run just the CTE to inspect.

# Basic CTE: Calculate call duration, then filter
# Syntax reminder: WITH cte_name AS ( ... ) SELECT ... FROM cte_name
print("=== Calls Over 8 Minutes (CTE) ===")
print(f"{'Call ID':<12} {'Channel':<12} {'Disposition':<14} {'Duration':>8}")
print("-" * 48)

results = cursor.execute("""
    WITH call_durations AS (
        -- Step 1: Calculate duration for every call (CTE = named temporary result)
        -- This runs first, before the outer SELECT even starts.
        SELECT call_id, channel, disposition,
               ROUND((julianday(call_ended_at) - julianday(call_started_at)) * 1440, 1) as duration_min
        FROM calls
        -- WHY not filter here? Keep the CTE general. Let the outer query decide what to filter.
    )
    -- Step 2: Now filter the named result — no repeated formula
    SELECT call_id, channel, disposition, duration_min
    FROM call_durations
    WHERE duration_min > 8   -- This is only possible because we named it in the CTE above
    ORDER BY duration_min DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<14} {row[3]:>7.1f}m")

print(f"\n{len(results)} calls over 8 minutes.")
print("\nThe CTE calculated duration once, then the outer query filtered on it.")
print("Without a CTE, you'd repeat the julianday formula in both SELECT and WHERE.")
# You should see roughly 10-12 calls — the longer completed calls


In [ ]:
# Multi-CTE pattern: each CTE is ONE step in the funnel.
# Think of it as building a spreadsheet where each tab feeds the next.
# The final SELECT just combines the pre-calculated steps.
# WHY this beats subqueries: you can read it top-to-bottom like a recipe.
# In dbt (the industry-standard SQL transformation tool), every model is structured exactly this way.

# Multi-CTE: Conversion funnel — Total → Completed → Ordered → Paid
print("=== Call Center Conversion Funnel ===")
print()

result = cursor.execute("""
    WITH total AS (
        -- CTE 1: Count all calls regardless of outcome (the top of the funnel)
        SELECT COUNT(*) as n FROM calls
    ),
    completed AS (
        -- CTE 2: Count only calls with 'completed' disposition
        -- WHY separate CTE? Keeps the calculation isolated and readable.
        SELECT COUNT(*) as n FROM calls WHERE disposition = 'completed'
    ),
    ordered AS (
        -- CTE 3: Count calls that led to at least one order
        -- COUNT DISTINCT because one call could theoretically have multiple orders
        SELECT COUNT(DISTINCT o.call_id) as n
        FROM orders o
    ),
    paid AS (
        -- CTE 4: Count calls where the order had an approved payment (bottom of funnel)
        -- This joins orders to payments to check payment status
        SELECT COUNT(DISTINCT o.call_id) as n
        FROM orders o
        JOIN payments p ON o.order_id = p.order_id
        WHERE p.status = 'approved'
    )
    -- Final SELECT: cross-join all 4 single-row CTEs to compute percentages
    -- FROM total t, completed c, ordered o, paid p = cartesian product of 1x1x1x1 = 1 row
    SELECT
        t.n as total_calls,
        c.n as completed,
        o.n as ordered,
        p.n as paid,
        ROUND(c.n * 100.0 / t.n, 1) as comp_pct,
        ROUND(o.n * 100.0 / c.n, 1) as order_pct,
        ROUND(p.n * 100.0 / o.n, 1) as pay_pct
    FROM total t, completed c, ordered o, paid p
""").fetchone()

print(f"  All Calls:       {result[0]:>4}          (100%)")
print(f"  Completed:       {result[1]:>4}  →  {result[4]}% of calls completed")
print(f"  Placed Order:    {result[2]:>4}  →  {result[5]}% of completed placed an order")
print(f"  Payment Approved:{result[3]:>4}  →  {result[6]}% of orders had approved payment")

print("\nEach CTE is one step in the funnel. The final SELECT combines them.")
print("This pattern is how you build analytics funnels in BigQuery, dbt, and dashboards.")

# Funnel by channel — same pattern but partitioned by channel
# WHY LEFT JOIN here? We want all calls in the funnel base, even those with no order/payment.
print("\n=== Funnel by Channel ===")
print(f"{'Channel':<12} {'Calls':>6} {'Comp':>6} {'Orders':>7} {'Paid':>6} {'End-to-End':>10}")
print("-" * 49)

results = cursor.execute("""
    WITH funnel AS (
        SELECT c.channel,
               COUNT(DISTINCT c.call_id) as total_calls,
               -- CASE WHEN inside COUNT DISTINCT: counts only rows where condition is true
               COUNT(DISTINCT CASE WHEN c.disposition = 'completed' THEN c.call_id END) as completed,
               COUNT(DISTINCT o.order_id) as ordered,
               COUNT(DISTINCT CASE WHEN p.status = 'approved' THEN o.order_id END) as paid
        FROM calls c
        LEFT JOIN orders o ON c.call_id = o.call_id    -- LEFT JOIN keeps calls with no order
        LEFT JOIN payments p ON o.order_id = p.order_id -- LEFT JOIN keeps orders with no payment
        GROUP BY c.channel
    )
    SELECT channel, total_calls, completed, ordered, paid,
           ROUND(paid * 100.0 / total_calls, 1) as end_to_end_pct  -- calls that made it all the way to paid
    FROM funnel
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>6} {row[3]:>7} {row[4]:>6} {row[5]:>9}%")
# You should see live_agent vs va side-by-side — which channel converts better end-to-end?


### Why CTEs instead of subqueries?

The same query above could be written as nested subqueries — but compare the readability:

```sql
-- Subquery version (hard to follow — reads inside-out):
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (...) as rn
    FROM (
        SELECT campaign, SUM(total) as revenue
        FROM (SELECT c.*, o.total FROM calls c LEFT JOIN orders o ON ...) t
        GROUP BY campaign
    ) agg
) ranked WHERE rn <= 3

-- CTE version (reads top-to-bottom, like a recipe):
WITH raw AS (SELECT c.*, o.total FROM calls c LEFT JOIN orders o ON ...),
agg AS (SELECT campaign, SUM(total) as revenue FROM raw GROUP BY campaign),
ranked AS (SELECT *, ROW_NUMBER() OVER (...) as rn FROM agg)
SELECT * FROM ranked WHERE rn <= 3
```

**Three practical reasons DEs always use CTEs:**
1. **Debug one step at a time** — comment out the final SELECT, run just the CTE to inspect intermediate results
2. **Reuse without repeating** — reference the CTE name multiple times in the final SELECT
3. **dbt uses CTEs natively** — every dbt model is a `WITH` block; learning CTEs now = learning dbt syntax


### CTEs — Gotchas & Interview Questions

**Common mistakes:**

| Mistake | What happens | Fix |
|---------|-------------|-----|
| Forgetting the comma between CTEs | SQL syntax error | Every CTE except the last needs a trailing comma |
| Referencing a CTE before it's defined | Error — CTEs are evaluated top-to-bottom | Move the CTE above the one that references it |
| Using a CTE name that matches a real table | Ambiguous — CTE shadows the real table | Use descriptive names: `call_metrics`, not `calls` |
| Putting a semicolon after a CTE | Ends the statement prematurely | Only one semicolon, at the very end of the final SELECT |

**Interview questions you will see:**

1. *"What is a CTE and when would you use it over a subquery?"*
   - CTE = Common Table Expression. Use it when: (a) you need to reuse a subquery more than once, (b) the query has more than 2 levels of nesting, (c) you want to debug a step in isolation.

2. *"Can a CTE reference another CTE?"*
   - Yes — that's the power of the multi-CTE pattern. `WITH a AS (...), b AS (SELECT * FROM a WHERE ...)` is valid and common.

3. *"Are CTEs materialized or re-evaluated each time?"*
   - In most engines (BigQuery, Postgres, Spark SQL): re-evaluated each time they're referenced. In some engines you can use `MATERIALIZED` to force a single computation. In dbt, CTEs are always re-evaluated unless you use `ref()` with materialization settings.

> **BEGINNER NOTE:** In dbt, every model file is structured exactly like a multi-CTE query. The file starts with `WITH`, defines named steps, and ends with a final `SELECT`. Learning CTEs now means you're already halfway to reading and writing dbt models.


---

## 5. Window Functions — The DE Power Tool

**One-line answer:** Window functions compute values across rows related to the current row — without collapsing the result into groups.

> **Analogy:** Think of a window function like a moving spotlight on a sorted list. For each row, the spotlight looks at neighboring rows and computes something (a rank, a running total, a gap), then moves to the next row. Unlike GROUP BY, which collapses the list into totals, the spotlight moves without erasing anything — every original row stays.

### GROUP BY vs. Window Functions

| Aspect | GROUP BY | Window Function |
|--------|---------|----------------|
| Output rows | One row per group | All original rows preserved |
| Syntax | `GROUP BY column` | `OVER (PARTITION BY ... ORDER BY ...)` |
| Use case | Totals, averages, counts | Rankings, running totals, row comparisons |

### The OVER Clause

```sql
function() OVER (
    PARTITION BY column    -- Optional: split into groups (like GROUP BY, but keeps rows)
    ORDER BY column        -- Optional: define row order within each partition
    ROWS BETWEEN ...       -- Optional: define the window frame
)
```

> `PARTITION BY campaign` means: reset the calculation for each campaign separately — as if you ran the window function once per campaign, then stitched the results together.

### Window Functions You'll Use as a DE

| Function | What It Does | DE Use Case |
|----------|-------------|-------------|
| `ROW_NUMBER()` | Unique sequential number per partition | Deduplication — keep only the latest record |
| `RANK()` | Rank with gaps for ties | Leaderboards, top-N analysis |
| `DENSE_RANK()` | Rank without gaps | When you need exactly N distinct ranks |
| `LAG(col, n)` | Value from n rows before | Compare to previous period |
| `LEAD(col, n)` | Value from n rows after | Look ahead (next event) |
| `SUM() OVER` | Running total | Cumulative revenue, progressive counts |
| `AVG() OVER` | Moving average | Smoothed trends, rolling metrics |
| `NTILE(n)` | Split into n equal buckets | Quartile/percentile analysis |

> **Why DEs love window functions:** They solve problems that are painful or impossible with GROUP BY — deduplication, running totals, period-over-period comparisons, and ranking. PySpark has the same window functions with identical syntax.

In [ ]:
# ROW_NUMBER() assigns a unique sequential integer to each row within a partition.
# PARTITION BY dnis means: restart numbering from 1 for each different DNIS.
# ORDER BY call_started_at means: number them in chronological order.
# WHY is this useful? The classic deduplication pattern:
#   WITH ranked AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY call_id ORDER BY updated_at DESC) as rn)
#   SELECT * FROM ranked WHERE rn = 1  ← keep only the most recent version of each record

# ROW_NUMBER: Number each call within its DNIS
print("=== ROW_NUMBER — Calls Numbered Per DNIS (Acme Spring Sale) ===")
print(f"{'#':>3} {'Call ID':<12} {'Started At':<24} {'Disposition':<14}")
print("-" * 55)

results = cursor.execute("""
    SELECT
        -- OVER (PARTITION BY dnis ORDER BY call_started_at):
        -- "Within each dnis group, number rows in time order"
        ROW_NUMBER() OVER (PARTITION BY dnis ORDER BY call_started_at) as call_num,
        call_id, call_started_at, disposition
    FROM calls
    WHERE dnis = '8005551234'  -- Filter to one DNIS to see the numbering clearly
    ORDER BY call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:>3} {row[1]:<12} {row[2]:<24} {row[3]:<14}")

print(f"\nROW_NUMBER assigned 1..{len(results)} within this DNIS.")
print("Use case: If call_001 appeared twice, ROW_NUMBER() helps you pick one (deduplication).")

# RANK vs DENSE_RANK: both rank rows but handle TIES differently
# RANK:       1, 2, 2, 4  (gap after tie — position 3 is skipped)
# DENSE_RANK: 1, 2, 2, 3  (no gap — next rank continues from 3)
# WHY does it matter? If you filter WHERE rank <= 3, RANK might return fewer than 3 rows on a tie.
# DENSE_RANK guarantees you get all rows at each rank level.
print("\n=== RANK vs DENSE_RANK — Campaigns by Revenue ===")
print(f"{'Campaign':<26} {'Revenue':>10} {'Rank':>5} {'Dense':>6}")
print("-" * 49)

results = cursor.execute("""
    WITH campaign_rev AS (
        -- Pre-calculate revenue per campaign (needed before we can rank it)
        SELECT s.campaign_name,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue  -- COALESCE: treat NULL as 0
        FROM sources s
        LEFT JOIN calls c ON s.dnis = c.dnis
        LEFT JOIN orders o ON c.call_id = o.call_id
        GROUP BY s.campaign_name
    )
    SELECT campaign_name, revenue,
           RANK() OVER (ORDER BY revenue DESC) as rnk,         -- gaps on ties
           DENSE_RANK() OVER (ORDER BY revenue DESC) as dense_rnk  -- no gaps on ties
    FROM campaign_rev
""").fetchall()

for row in results:
    print(f"{row[0]:<26} ${row[1]:>9.2f} {row[2]:>5} {row[3]:>6}")
# You should see 6 rows. Note where RANK and DENSE_RANK diverge — that's where ties occurred.


### LAG and LEAD — Looking Backward and Forward

`LAG(column, n)` returns the value from `n` rows before the current row.  
`LEAD(column, n)` returns the value from `n` rows after.

```sql
-- Time between consecutive calls on the same DNIS
SELECT call_id,
       call_started_at,
       LAG(call_started_at) OVER (PARTITION BY dnis ORDER BY call_started_at) as prev_call
FROM calls
```

> **DE use case:** LAG is how you calculate period-over-period changes — "How does today's call volume compare to yesterday?" You'll use this pattern in BigQuery marts and PySpark.

In [ ]:
# LAG() looks BACKWARD: returns the value from a previous row.
# LEAD() looks FORWARD: returns the value from a future row.
# Both require ORDER BY inside OVER() to define what "previous" and "next" mean.
# WHY use LAG for time gaps? It's the standard pattern for calculating elapsed time
# between sequential events — call center gaps, pipeline run intervals, session durations.

# LAG: Time between consecutive calls on Acme Spring Sale DNIS
print("=== LAG — Time Between Calls (Acme Spring Sale) ===")
print(f"{'Call ID':<12} {'Started At':<24} {'Prev Call':<24} {'Gap':>6}")
print("-" * 68)

results = cursor.execute("""
    SELECT call_id, call_started_at,
           -- LAG(column) OVER (ORDER BY ...): returns the value from the PREVIOUS row
           -- The first row returns NULL because there is no row before it
           LAG(call_started_at) OVER (ORDER BY call_started_at) as prev_call,
           ROUND(
               -- Calculate minutes between this call's start and the previous call's start
               -- julianday difference gives elapsed days → multiply by 1440 to get minutes
               (julianday(call_started_at) -
                julianday(LAG(call_started_at) OVER (ORDER BY call_started_at))) * 1440,
               0
           ) as gap_minutes  -- NULL for the first row (no previous call to measure from)
    FROM calls
    WHERE dnis = '8005551234'   -- Filter to one campaign to keep output readable
    ORDER BY call_started_at
""").fetchall()

for row in results:
    prev = row[2] if row[2] else '(first call)'  # NULL = first call, no previous exists
    gap = f"{int(row[3])}m" if row[3] else '-'   # NULL gap for first row
    print(f"{row[0]:<12} {row[1]:<24} {prev:<24} {gap:>6}")
# You should see: first row has no gap, subsequent rows show 30-60+ minute gaps

# LEAD() is LAG() in reverse — looks at the NEXT row instead of the previous.
# Use case: "After a dropped call, what did the customer do next?" (callback analysis)
print("\n=== LEAD — Next Call Disposition ===")
print(f"{'Call ID':<12} {'Disposition':<14} {'Next Disp':<14}")
print("-" * 40)

results = cursor.execute("""
    SELECT call_id, disposition,
           -- LEAD returns the NEXT row's value. Last row returns NULL (no next row).
           LEAD(disposition) OVER (ORDER BY call_started_at) as next_disposition
    FROM calls
    WHERE dnis = '8005551234'
    ORDER BY call_started_at
""").fetchall()

for row in results:
    next_d = row[2] if row[2] else '(last call)'  # NULL = no next call
    print(f"{row[0]:<12} {row[1]:<14} {next_d:<14}")
# BEGINNER NOTE: Look for 'dropped' followed by 'completed' — that's a customer who called back.


### Running Totals and Moving Averages

```sql
-- Running total: sum all rows from the start up to the current row
SUM(amount) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING)

-- Moving average: average of the current row and 2 preceding rows
AVG(amount) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
```

| Frame Clause | Meaning |
|-------------|--------|
| `ROWS UNBOUNDED PRECEDING` | From first row to current row |
| `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` | Current row + 2 rows before |
| `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING` | Current row + 1 before + 1 after |

> **Translation:** `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` means "from the very first row up to and including the current row" — this creates a running total.

> **DE use case:** Running totals show cumulative revenue. Moving averages smooth out daily spikes in call volume — essential for dashboards and trend analysis.

In [ ]:
# SUM() OVER (ORDER BY ... ROWS UNBOUNDED PRECEDING) = running total.
# "ROWS UNBOUNDED PRECEDING" means: include every row from the very first row up to this one.
# As you move down the result set, the window expands — each row adds its value to the total.
# WHY running totals matter: cumulative revenue charts, progress-to-quota dashboards,
# pipeline load tracking — all require this pattern.

# Running total of revenue from orders
print("=== Running Total Revenue ===")
print(f"{'Order':<10} {'Call':<12} {'Amount':>8} {'Running Total':>14} {'Mov Avg (3)':>12}")
print("-" * 58)

results = cursor.execute("""
    SELECT o.order_id, o.call_id, o.total,
           -- Running total: sum from first order to this order, sorted by call time
           ROUND(SUM(o.total) OVER (ORDER BY c.call_started_at
                ROWS UNBOUNDED PRECEDING), 2) as running_total,
           -- Moving average over 3 orders: current row + 2 rows before it
           -- WHY moving avg? Smooths out spikes — a $168 order after two $50 orders
           -- averages to ~$89, showing the trend rather than the outlier.
           ROUND(AVG(o.total) OVER (ORDER BY c.call_started_at
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) as moving_avg_3
    FROM orders o
    JOIN calls c ON o.call_id = c.call_id  -- JOIN (not LEFT JOIN) — every order must have a call
    ORDER BY c.call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:<10} {row[1]:<12} ${row[2]:>7.2f} ${row[3]:>13.2f} ${row[4]:>11.2f}")
# You should see running_total growing with each order, moving_avg_3 fluctuating around the mean

print(f"\nFinal running total: ${results[-1][3]:,.2f}")
print("The moving average smooths out individual order variation.")


In [ ]:
# This query uses TWO separate RANK() calls in the same SELECT — ranking by different criteria.
# WHY? Because the campaign with the most revenue is not always the best converter.
# By ranking both, the business can see trade-offs: which campaigns are high-volume vs high-rate?
# BEGINNER NOTE: Each OVER() clause is independent. Two RANK() calls = two separate window passes.

# Campaign performance dashboard using multiple window functions
print("=== Campaign Performance Dashboard (Window Functions) ===")
print(f"{'Campaign':<26} {'Revenue':>9} {'Rev Rank':>9} {'Conv%':>7} {'Conv Rank':>10}")
print("-" * 63)

results = cursor.execute("""
    WITH campaign_stats AS (
        -- Step 1: Aggregate base metrics per campaign
        SELECT s.campaign_name, s.channel,
               COUNT(DISTINCT c.call_id) as total_calls,
               COUNT(DISTINCT o.order_id) as orders,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue  -- COALESCE: no orders = $0, not NULL
        FROM calls c
        JOIN sources s ON c.dnis = s.dnis
        LEFT JOIN orders o ON c.call_id = o.call_id
        GROUP BY s.campaign_name, s.channel
    )
    -- Step 2: Apply TWO independent window functions on the aggregated data
    SELECT campaign_name,
           revenue,
           RANK() OVER (ORDER BY revenue DESC) as rev_rank,       -- Rank by total revenue
           ROUND(orders * 100.0 / total_calls, 1) as conv_rate,  -- Conversion rate calculation
           RANK() OVER (ORDER BY orders * 100.0 / total_calls DESC) as conv_rank  -- Rank by conversion rate
    FROM campaign_stats
    ORDER BY revenue DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} ${row[1]:>8.2f} {row[2]:>9} {row[3]:>6}% {row[4]:>10}")

print("\nNotice: highest revenue doesn't always mean best conversion rate.")
print("Window functions let you rank by multiple criteria in the same query.")
# You should see some campaigns flip positions between rev_rank and conv_rank.


### Why window functions instead of GROUP BY?

They solve completely different problems. Here's the key distinction:

**GROUP BY collapses rows — window functions preserve them.**

```sql
-- GROUP BY: returns ONE row per campaign (detail is gone)
SELECT campaign_name, SUM(revenue) as total
FROM orders GROUP BY campaign_name

-- Window function: returns EVERY order row, but adds the campaign total alongside it
SELECT order_id, campaign_name, revenue,
       SUM(revenue) OVER (PARTITION BY campaign_name) as campaign_total
FROM orders
-- ↑ You get the individual order AND the group total on the same row
```

**Three things GROUP BY cannot do — but window functions can:**

1. **Deduplication** — `ROW_NUMBER() OVER (PARTITION BY call_id ORDER BY updated_at DESC)` keeps the latest version of each record. GROUP BY can't do this without a subquery.
2. **Running totals** — `SUM(revenue) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING)`. GROUP BY gives you totals per group, not cumulative totals.
3. **Period-over-period comparison** — `LAG(revenue, 1) OVER (ORDER BY date)` compares today to yesterday on the same row. Impossible with GROUP BY alone.

> **Interview tip:** If an interviewer asks "how would you deduplicate a table keeping only the most recent record per ID?" — the answer is always `ROW_NUMBER() OVER (PARTITION BY id ORDER BY updated_at DESC)` followed by `WHERE rn = 1`.


### Window Functions — Gotchas & Interview Questions

**Common mistakes:**

| Mistake | What happens | Fix |
|---------|-------------|-----|
| Forgetting `ORDER BY` inside `OVER()` for running totals | Result is undefined — database can return rows in any order | Always include `ORDER BY` when order matters |
| Using window function in `WHERE` | SQL error — window functions run after WHERE | Wrap in a CTE or subquery, then filter |
| Mixing `GROUP BY` and window functions carelessly | Window function sees only the grouped rows, not originals | Apply window functions in a second CTE after GROUP BY |
| `RANK()` vs `ROW_NUMBER()` for deduplication | RANK gives same number to ties — two rows both get rank 1 | Use `ROW_NUMBER()` for deduplication (always unique) |

**Interview questions you will see:**

1. *"Find the second highest salary in a table."*
   - Answer: `DENSE_RANK() OVER (ORDER BY salary DESC)` then filter `WHERE dense_rnk = 2`

2. *"Deduplicate a table keeping only the most recent record per user."*
   - Answer: `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC)` then `WHERE rn = 1`

3. *"Calculate a 7-day moving average of daily revenue."*
   - Answer: `AVG(revenue) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`

4. *"For each call, show the previous call's disposition on the same row."*
   - Answer: `LAG(disposition, 1) OVER (PARTITION BY dnis ORDER BY call_started_at)`

> **BEGINNER NOTE:** Window functions are the #1 topic in Data Engineer SQL interviews. Every company that processes time-series data (which is all of them) will ask you about LAG, ROW_NUMBER, and running totals.


---

## 6. Date/Time Mastery for Pipelines

**One-line answer:** Every pipeline bug you'll encounter involves dates, timezones, or both.

In M02, you saw the UTC bug — evening EST calls appearing as the next day in UTC. Now let's build the reporting queries that handle this correctly.

### SQLite Date/Time Functions

| Function | Example | Result |
|----------|---------|--------|
| `DATE(ts)` | `DATE('2026-03-10T19:00:00Z')` | `2026-03-10` |
| `TIME(ts)` | `TIME('2026-03-10T19:00:00Z')` | `19:00:00` |
| `STRFTIME('%H', ts)` (String Format Time — converts a datetime to a formatted string) | Extract hour | `19` |
| `STRFTIME('%w', ts)` | Day of week (0=Sun) | `2` |
| `STRFTIME('%Y-%m', ts)` | Year-month | `2026-03` |
| `DATE(ts, '-5 hours')` | UTC to EST | Shifts timestamp |
| `DATE(ts, '+1 day')` | Add 1 day | Next day |
| `julianday(ts)` | Julian day number | For arithmetic |

> **In production:** BigQuery uses `TIMESTAMP_SUB()` and `FORMAT_TIMESTAMP()`. PySpark uses `from_utc_timestamp()`. The concept is the same — always convert to the reporting timezone before aggregating by date.

In [ ]:
# STRFTIME = String Format Time — converts a datetime value into a formatted string.
# '%H' extracts the hour as a zero-padded 2-digit string ('08', '13', '22', etc.)
# CAST(...AS INTEGER) converts '08' → 8 so we can sort numerically (not alphabetically).
# WHY subtract 5 hours? All timestamps are stored in UTC ('Z' suffix = UTC).
# EST = UTC - 5 hours. If we don't convert, an 8pm EST call appears as 1am UTC next day.

# Hourly call volume (EST) — when do calls peak?
print("=== Hourly Call Volume (EST) ===")
print(f"{'Hour (EST)':>10} {'Calls':>6} {'Completed':>10} {'Comp%':>7} {'Bar'}")
print("-" * 50)

results = cursor.execute("""
    SELECT CAST(STRFTIME('%H', call_started_at, '-5 hours') AS INTEGER) as hour_est,
           -- ↑ Extract hour (0-23) after shifting from UTC to EST
           COUNT(*) as calls,
           SUM(CASE WHEN disposition = 'completed' THEN 1 ELSE 0 END) as completed,
           ROUND(SUM(CASE WHEN disposition = 'completed' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 0) as comp_pct
           -- ↑ WHY 1.0 not 1? Forces floating-point division. Integer / integer = integer (truncated).
    FROM calls
    GROUP BY CAST(STRFTIME('%H', call_started_at, '-5 hours') AS INTEGER)
    ORDER BY hour_est
""").fetchall()

for row in results:
    bar = '█' * row[1]        # Visual bar chart — one block per call
    ampm = 'AM' if row[0] < 12 else 'PM'
    h = row[0] if row[0] <= 12 else row[0] - 12
    h = 12 if h == 0 else h   # Convert 0 → 12 for 12 AM display
    print(f"{h:>6} {ampm}  {row[1]:>6} {row[2]:>10} {row[3]:>6}% {bar}")

print("\nThis is how you build mart_hourly_volume — the gold-layer mart.")
print("Notice: all times converted to EST before grouping.")
# You should see calls concentrated in the 8AM-5PM EST window


In [ ]:
# DATE(timestamp, '-5 hours'): SQLite modifier that subtracts 5 hours from the timestamp.
# This converts UTC storage to EST reporting time BEFORE we GROUP BY date.
# WHY must conversion happen BEFORE GROUP BY? If you GROUP BY DATE(call_started_at)
# (no conversion), evening EST calls (e.g. 8pm EST = 1am UTC next day) land on the wrong date.
# The UTC bug doesn't crash your pipeline — it silently puts data on the wrong day.

# Daily metrics with proper EST conversion
print("=== Daily Campaign Report (EST) ===")
print(f"{'Date (EST)':<12} {'Calls':>6} {'Comp':>6} {'Orders':>7} {'Revenue':>10} {'Conv%':>7}")
print("-" * 50)

results = cursor.execute("""
    SELECT DATE(c.call_started_at, '-5 hours') as call_date_est,  -- Convert to EST before grouping
           COUNT(DISTINCT c.call_id) as calls,
           SUM(CASE WHEN c.disposition = 'completed' THEN 1 ELSE 0 END) as completed,
           COUNT(DISTINCT o.order_id) as orders,
           ROUND(COALESCE(SUM(o.total), 0), 2) as revenue,
           ROUND(COUNT(DISTINCT o.order_id) * 100.0 / COUNT(DISTINCT c.call_id), 1) as conv_pct
    FROM calls c
    LEFT JOIN orders o ON c.call_id = o.call_id  -- LEFT JOIN keeps calls with no order
    GROUP BY DATE(c.call_started_at, '-5 hours')  -- Group AFTER conversion, not before
    ORDER BY call_date_est
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>6} {row[3]:>7} ${row[4]:>9.2f} {row[5]:>6}%")

# Compare UTC vs EST totals to prove the bug
# WHY: calls starting at 7pm-11:59pm EST have UTC timestamps on the NEXT calendar day.
# Without conversion: those calls appear in tomorrow's report. With conversion: correct.
print("\n=== UTC vs EST Daily Totals — Proving the Bug ===")
print(f"{'Date':<12} {'UTC Count':>10} {'EST Count':>10} {'Diff':>6}")
print("-" * 40)

results = cursor.execute("""
    WITH utc_counts AS (
        -- Count calls grouped by RAW UTC date (no timezone conversion)
        SELECT DATE(call_started_at) as dt, COUNT(*) as n FROM calls GROUP BY DATE(call_started_at)
    ),
    est_counts AS (
        -- Count calls grouped by CONVERTED EST date
        SELECT DATE(call_started_at, '-5 hours') as dt, COUNT(*) as n FROM calls GROUP BY DATE(call_started_at, '-5 hours')
    )
    SELECT COALESCE(u.dt, e.dt) as dt,  -- COALESCE handles dates that appear in one but not the other
           COALESCE(u.n, 0) as utc_n,
           COALESCE(e.n, 0) as est_n,
           COALESCE(e.n, 0) - COALESCE(u.n, 0) as diff  -- Positive diff = EST has more on this date
    FROM utc_counts u
    LEFT JOIN est_counts e ON u.dt = e.dt
    ORDER BY dt
""").fetchall()

for row in results:
    flag = ' <- wrong!' if row[3] != 0 else ''
    print(f"{row[0]:<12} {row[1]:>10} {row[2]:>10} {row[3]:>+6}{flag}")

print("\nAlways convert to reporting timezone BEFORE grouping by date.")
# You should see a diff on 2026-03-10 (UTC has more) and 2026-03-11 (EST has more)
# That's the UTC bug: late-evening EST calls "moved" to the next UTC day


---

## 7. Data Transformation Patterns

**One-line answer:** Transformation is the T in ETL (Extract, Transform, Load) — turning raw data into clean, enriched, analysis-ready data.

### CASE WHEN for Categorization

You already used CASE WHEN for pivoting. It's also how you create derived columns — categorizing raw values into business-meaningful buckets.

```sql
-- Categorize call duration
CASE
    WHEN duration_min < 3 THEN 'Short'
    WHEN duration_min < 8 THEN 'Medium'
    ELSE 'Long'
END as duration_category
```

### NULL Handling


> **Analogy:** NULL is not zero and not empty string — it means *unknown*. Imagine asking for someone's middle name when they don't have one. Writing a blank (`''`) means they have an empty middle name. NULL means we simply don't know. And here's the trap: adding unknown to any number gives unknown — a single NULL in a SUM silently makes the whole result NULL.

| Function | What It Does | Example |
|----------|-------------|--------|
| `COALESCE(a, b, c)` | First non-NULL value | `COALESCE(auth_code, 'N/A')` |
| `NULLIF(a, b)` | Returns NULL if a = b | `NULLIF(amount, 0)` — prevents divide-by-zero |
| `IFNULL(a, b)` | SQLite-specific COALESCE for 2 args | `IFNULL(sentiment, 'unknown')` |

> **In production:** NULL handling is where most data quality bugs hide. A single NULL in a SUM makes the whole result NULL unless you use COALESCE. Always handle NULLs explicitly.

In [ ]:
# CASE WHEN for categorization: turn a continuous number (duration) into discrete buckets.
# WHY: Raw numbers are hard to summarize. Buckets turn "6.7 minutes" into "Medium" —
# a category that's meaningful to a business analyst or ops manager.
# The numbers in the CASE WHEN (3, 8) are business thresholds — not arbitrary.
# Short (<3 min) calls are usually drops. Long (>8 min) calls often indicate complex issues.

# Categorize call duration + revenue tier
print("=== Call Duration Categories ===")
print(f"{'Category':<20} {'Count':>6} {'Avg Duration':>12} {'Completed%':>11}")
print("-" * 51)

results = cursor.execute("""
    WITH call_dur AS (
        -- Step 1: Calculate raw duration for every call
        SELECT *,
               ROUND((julianday(call_ended_at) - julianday(call_started_at)) * 1440, 1) as dur_min
        FROM calls
        -- WHY CTE here? We use dur_min in both CASE WHEN and AVG — define once, reuse twice.
    )
    SELECT
        CASE
            WHEN dur_min < 3 THEN '1. Short (<3 min)'    -- Likely dropped or IVR-only
            WHEN dur_min < 8 THEN '2. Medium (3-8 min)'  -- Standard handled call
            ELSE '3. Long (>8 min)'                       -- Complex issue or high-value sale
        END as category,
        COUNT(*) as cnt,
        ROUND(AVG(dur_min), 1) as avg_dur,
        -- Completion rate per category: how often does each duration bucket result in 'completed'?
        ROUND(SUM(CASE WHEN disposition = 'completed' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) as comp_pct
    FROM call_dur
    GROUP BY category  -- Group by the CASE WHEN result (the category label, not the number)
    ORDER BY category  -- Alphabetical: 1. Short, 2. Medium, 3. Long
""").fetchall()

for row in results:
    print(f"{row[0]:<20} {row[1]:>6} {row[2]:>10.1f} min {row[3]:>10}%")

print("\nInsight: Longer calls tend to have higher completion rates.")
print("Dropped calls are almost always short — the customer hung up quickly.")
# You should see: Short calls have low comp%, Long calls have high comp%
# BEGINNER NOTE: This is why call centers track Average Handle Time (AHT) by outcome.


In [ ]:
# COALESCE(value, fallback): returns the first non-NULL argument.
# If auth_code is NULL (declined payment), COALESCE returns '(declined)' instead.
# WHY not just display NULL? NULL displays as blank or 'None' in output — confusing for readers.
# COALESCE makes the intent explicit: "we know there's no auth code, and we're saying so."

# NULL handling patterns
print("=== COALESCE: Declined Payments with Default Auth Code ===")
print(f"{'Txn ID':<10} {'Order':<10} {'Auth Code':<12} {'Amount':>8} {'Status':<10}")
print("-" * 52)

results = cursor.execute("""
    SELECT transaction_id, order_id,
           -- COALESCE: if auth_code IS NULL, return '(declined)' instead
           -- This makes declined payments readable instead of showing blank
           COALESCE(auth_code, '(declined)') as auth_display,
           amount, status
    FROM payments
    WHERE status = 'declined'  -- Only show the 2 declined payments for this demo
""").fetchall()

for row in results:
    print(f"{row[0]:<10} {row[1]:<10} {row[2]:<12} ${row[3]:>7.2f} {row[4]:<10}")
# You should see 2 rows — txn_006 and txn_012 — both with '(declined)' where auth_code was NULL

# NULLIF(a, b): returns NULL if a equals b, otherwise returns a.
# WHY use NULLIF for division? Dividing by zero crashes the query (ZeroDivisionError in Python,
# SQL error in most databases). NULLIF(count, 0) converts zero → NULL, and NULL / anything = NULL.
# Then you can COALESCE the NULL result to 0 if needed for display.
print("\n=== NULLIF: Safe Division (Prevents Divide-by-Zero) ===")
print(f"{'Campaign':<26} {'Calls':>6} {'Orders':>7} {'Conv%':>7}")
print("-" * 48)

results = cursor.execute("""
    SELECT s.campaign_name,
           COUNT(DISTINCT c.call_id) as calls,
           COUNT(DISTINCT o.order_id) as orders,
           -- NULLIF(COUNT(DISTINCT c.call_id), 0): if somehow calls = 0, return NULL instead
           -- This prevents division by zero. In production, a new campaign with no calls yet
           -- would cause a crash without NULLIF.
           ROUND(COUNT(DISTINCT o.order_id) * 100.0 /
                 NULLIF(COUNT(DISTINCT c.call_id), 0), 1) as conv_pct
    FROM sources s
    LEFT JOIN calls c ON s.dnis = c.dnis     -- LEFT JOIN starts from sources (all 6 campaigns)
    LEFT JOIN orders o ON c.call_id = o.call_id
    GROUP BY s.campaign_name
    ORDER BY conv_pct DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:>6} {row[2]:>7} {row[3]:>6}%")

print("\nNULLIF(x, 0) returns NULL when x is 0, so division yields NULL instead of an error.")
print("COALESCE(result, 0) can then turn that NULL into 0 for display.")
# BEGINNER NOTE: In real pipelines, divide-by-zero from empty partitions is a common silent bug.
# Always use NULLIF when dividing by a COUNT, SUM, or other value that could be zero.


---

## 8. Query Optimization & Execution Plans

**One-line answer:** An execution plan tells you HOW the database will run your query — and where it's wasting time.

> **Analogy:** An execution plan is like the route your GPS calculates before driving — it tells you which roads (operations) it will take and where the traffic (expensive steps) is.

### Why DEs Care About Optimization

| Context | Impact of Slow Query |
|---------|--------------------|
| BigQuery | Scans more data = higher cost (pay per byte scanned) |
| PySpark | Unoptimized joins = shuffles = OOM (Out of Memory) errors — the program crashes because it ran out of RAM |
| Airflow DAG (Directed Acyclic Graph — a pipeline workflow where tasks run in dependency order) | Slow query = delayed pipeline = stale dashboards |
| Reports | Users waiting 30 seconds vs 2 seconds |

> **Analogy:** An index on a column is like the index at the back of a textbook — instead of reading every page to find "Hadoop," you flip to the index and go directly to page 247.

### EXPLAIN QUERY PLAN (SQLite)

Every database has an EXPLAIN command. SQLite's version:

```sql
EXPLAIN QUERY PLAN SELECT * FROM calls WHERE dnis = '8005551234'
```

Key terms in the output:

| Term | Meaning | Good or Bad? |
|------|---------|-------------|
| `SCAN` | Reads every row in the table | Slow on large tables |
| `SEARCH` | Uses an index to find rows | Fast — skips irrelevant rows |
| `USING INDEX` | Which index it's using | Good — means an index exists |
| `USING COVERING INDEX` | Index has all needed columns | Best — doesn't even touch the table |

In [ ]:
# EXPLAIN QUERY PLAN is a diagnostic command — it does NOT run the query.
# It tells you HOW the database engine plans to execute the query.
# WHY care? On 40 rows, every query is instant. On 40 million rows, a bad plan costs
# minutes of compute time or hundreds of dollars in BigQuery scan costs.
# Reading execution plans is how senior DEs catch performance problems before they hit production.

# EXPLAIN QUERY PLAN: Before and after indexing
print("=== WITHOUT Index ===")
# SCAN = the database reads EVERY row in the table to find matches.
# On 40 rows this is fine. On 40 million rows, this is a full table scan — very slow.
plan = cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT call_id, disposition FROM calls WHERE dnis = '8005551234'
""").fetchall()
for row in plan:
    print(f"  {row[-1]}")
# You should see: SCAN calls — meaning it reads all 40 rows to find dnis matches

# CREATE INDEX: build a lookup structure on the dnis column.
# Analogy: like adding an entry to a book's index — instead of reading every page,
# you jump directly to the relevant pages.
# Syntax: CREATE INDEX idx_<table>_<column> ON table(column)
cursor.execute("CREATE INDEX idx_calls_dnis ON calls(dnis)")

print("\n=== WITH Index on calls(dnis) ===")
plan = cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT call_id, disposition FROM calls WHERE dnis = '8005551234'
""").fetchall()
for row in plan:
    print(f"  {row[-1]}")
# You should see: SEARCH calls USING INDEX idx_calls_dnis — much faster than SCAN

# Create indexes on other commonly filtered/joined columns
# WHY these columns? disposition and channel appear in WHERE clauses.
# call_id in orders is a JOIN key — indexing it speeds up every JOIN to calls.
cursor.execute("CREATE INDEX idx_calls_disposition ON calls(disposition)")
cursor.execute("CREATE INDEX idx_calls_channel ON calls(channel)")
cursor.execute("CREATE INDEX idx_orders_call_id ON orders(call_id)")

print("\n=== JOIN with Index ===")
plan = cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT c.call_id, o.total
    FROM calls c
    JOIN orders o ON c.call_id = o.call_id   -- This JOIN benefits from idx_orders_call_id
    WHERE c.disposition = 'completed'         -- This filter benefits from idx_calls_disposition
""").fetchall()
for row in plan:
    print(f"  {row[-1]}")

print("\nSCAN = reads every row (slow on millions of rows)")
print("SEARCH = uses index to jump directly to matching rows (fast)")
# BEGINNER NOTE: Always create indexes on: JOIN keys (foreign keys), WHERE filter columns,
# ORDER BY columns on large tables. Don't over-index — writes become slower with each index.


### SQL Anti-Patterns That Kill Performance

| Anti-Pattern | Why It's Slow | Better Alternative |
|-------------|--------------|-------------------|
| `SELECT *` | Reads all columns, even unused ones | `SELECT col1, col2` — only what you need |
| `WHERE UPPER(name) = 'ACME'` | Function on column prevents index use | Store normalized, or use case-insensitive collation |
| `WHERE amount != 0` | Negative conditions scan everything | `WHERE amount > 0` — narrower scan |
| `OR` in WHERE | Often prevents index use | `UNION ALL` of two indexed queries |
| `SELECT DISTINCT` on large results | Sorts entire result to deduplicate | Fix the JOIN that's creating duplicates |
| Subquery in SELECT | Runs once per row | JOIN instead |
| `LIKE '%search%'` | Leading wildcard = full scan | Full-text search, or `LIKE 'search%'` |

> **SARGable queries:** A query is "SARGable" (Search ARGument able) when the WHERE clause can use an index. Keep columns bare — don't wrap them in functions.

```sql
-- NOT SARGable (can't use index):
WHERE STRFTIME('%Y', call_started_at) = '2026'

-- SARGable (can use index):
WHERE call_started_at >= '2026-01-01' AND call_started_at < '2027-01-01'
```

In [ ]:
# This cell proves the performance difference with real timing.
# WHY generate 10,000 rows? 40 rows is too fast to measure — everything is instant.
# 10,000 rows + 100 repeated queries makes the timing difference visible.
# In production, think 10 million rows — the speedup is the same ratio, but in minutes vs seconds.

# Performance demonstration with larger dataset
import time
import random

# Generate 10,000 calls for performance testing
# CREATE TABLE ... AS SELECT ... WHERE 1=0: copy the schema (columns) but no data
# WHY 1=0? The WHERE clause is always false — no rows are copied, just the table definition.
cursor.execute("CREATE TABLE calls_large AS SELECT * FROM calls WHERE 1=0")

dnis_list = ['8005551234', '8005551235', '8005552345', '8005552346', '8005553456', '8005553457']
dispositions = ['completed'] * 60 + ['dropped'] * 25 + ['transferred'] * 15
# WHY weighted list? 60% completed / 25% dropped / 15% transferred mimics real call center ratios.
channels = {'8005551234': 'live_agent', '8005551235': 'va', '8005552345': 'live_agent',
            '8005552346': 'va', '8005553456': 'live_agent', '8005553457': 'va'}

large_data = []
for i in range(10000):
    dnis = random.choice(dnis_list)
    hour = random.randint(8, 23)
    minute = random.randint(0, 59)
    day = random.randint(1, 28)
    dur = random.randint(60, 900)  # 1-15 min in seconds
    large_data.append((
        f"lg_{i:05d}", dnis, f"555{random.randint(1000000,9999999)}",
        f"2026-03-{day:02d}T{hour:02d}:{minute:02d}:00Z",
        f"2026-03-{day:02d}T{hour:02d}:{minute + dur//60:02d}:{dur%60:02d}Z",
        random.choice(dispositions),
        random.choice(['positive', 'neutral', 'negative']),
        channels[dnis]
    ))
# executemany with a list is MUCH faster than looping cursor.execute() — batched insert
cursor.executemany("INSERT INTO calls_large VALUES (?, ?, ?, ?, ?, ?, ?, ?)", large_data)
conn.commit()

# Benchmark WITHOUT index: time 100 identical queries
# WHY 100 queries? One query might be too fast to measure accurately on small data.
start = time.time()
for _ in range(100):
    cursor.execute("SELECT COUNT(*) FROM calls_large WHERE dnis = '8005551234'").fetchone()
no_idx = time.time() - start

# Create index on dnis — same as we did on the small table earlier
cursor.execute("CREATE INDEX idx_large_dnis ON calls_large(dnis)")

# Benchmark WITH index: same 100 queries, now with index available
start = time.time()
for _ in range(100):
    cursor.execute("SELECT COUNT(*) FROM calls_large WHERE dnis = '8005551234'").fetchone()
with_idx = time.time() - start

print(f"=== Performance: 10,000 rows, 100 queries ===")
print(f"  Without index: {no_idx:.4f}s")
print(f"  With index:    {with_idx:.4f}s")
print(f"  Speedup:       {no_idx/with_idx:.1f}x faster")
print(f"\nOn 10 million rows, the difference is minutes vs milliseconds.")
print("In BigQuery, it's also dollars — unindexed scans cost more.")
# You should see at least 2-5x speedup even on 10K rows. On 10M+ rows: 100-1000x.


### Why does an index make queries faster?

Without an index, a database has no choice: to find all rows where `dnis = '8005551234'`, it reads every single row in the table — even if only 10 out of 10 million match. This is called a **full table scan**.

An index is a separate data structure (a B-tree) that the database maintains alongside the table. Think of it as the index at the back of a textbook:

```
Index on calls(dnis):
  8005551234 → [row 1, row 8, row 15, row 21, ...]
  8005551235 → [row 6, row 13, row 18, ...]
  8005552345 → [row 3, row 9, row 16, ...]
  ...
```

With this structure, finding `dnis = '8005551234'` means:
1. Look up `8005551234` in the index (fast — B-tree lookup is O(log n))
2. Jump directly to those specific rows
3. Return results without reading any other rows

**The tradeoff:** Indexes speed up reads but slow down writes. Every INSERT, UPDATE, or DELETE also has to update the index. On a write-heavy pipeline (millions of inserts per hour), too many indexes can hurt throughput.

**Rule of thumb:** Index columns that appear in `WHERE`, `JOIN ON`, or `ORDER BY` in your most-run queries. Don't add indexes speculatively — use EXPLAIN to confirm they're being used.


### Query Optimization — Gotchas & Interview Questions

**Common mistakes:**

| Mistake | Performance impact | Fix |
|---------|------------------|-----|
| `SELECT *` in production queries | Scans all columns, even unused ones. In BigQuery: costs money per byte scanned | List only the columns you need |
| `WHERE UPPER(column) = 'VALUE'` | Function on column = index can't be used = full scan | Store data normalized, or use case-insensitive collation |
| `WHERE column != 'x'` | Negative conditions often skip indexes | Restructure with positive conditions where possible |
| Filtering on a calculated column without CTE | Recalculates for every row | Pre-calculate in a CTE, filter on the result |
| `LIKE '%word%'` (leading wildcard) | Full table scan — index can't skip ahead | Use `LIKE 'word%'` (no leading wildcard) or full-text search |

**Interview questions you will see:**

1. *"What does EXPLAIN QUERY PLAN tell you?"*
   - It shows the execution strategy: which indexes are used, whether joins are hash joins or nested loops, and the estimated row counts at each step. SCAN = slow, SEARCH = fast.

2. *"What is a SARGable query?"*
   - SARGable (Search ARGument able) means the WHERE clause can use an index. Bare columns in WHERE are SARGable. Wrapped columns (`WHERE YEAR(date) = 2026`) are not — use range comparisons instead (`WHERE date >= '2026-01-01' AND date < '2027-01-01'`).

3. *"When would you NOT create an index?"*
   - On columns with very low cardinality (e.g., a boolean column with only TRUE/FALSE — the index isn't selective enough to help). On small tables (full scan is fast anyway). On heavily write-intensive tables (every write updates every index).

4. *"What's the difference between a clustered and non-clustered index?"*
   - Clustered: the table rows are physically sorted by the index key (BigQuery uses clustering; Postgres uses `CLUSTER`). Non-clustered: a separate structure that points to rows. BigQuery's `CLUSTER BY` and `PARTITION BY` are the production equivalents of what we're doing with `CREATE INDEX` here.


---

## 9. Incremental Loading Strategies

**One-line answer:** Full refresh rebuilds everything every run. Incremental loading only processes what's new.

> **Analogy:** Think of a watermark like a bookmark in a book. Every time you finish reading, you move the bookmark to where you stopped. Next time, you start from the bookmark — you don't re-read the whole book. The pipeline works the same way: it remembers the last record it processed and only picks up new ones.

### Full Refresh vs. Incremental

| Aspect | Full Refresh | Incremental |
|--------|-------------|-------------|
| What it does | DROP + recreate the entire table | INSERT/UPDATE only new or changed rows |
| Speed | Slow on large datasets | Fast — processes only the delta |
| Complexity | Simple — just re-run the query | More complex — need a watermark |
| Data loss risk | Reprocessing errors affect all data | Errors affect only new data |
| When to use | Small tables, early development | Large tables, production pipelines |

### The Watermark Pattern

Track the last successfully processed timestamp. On the next run, only process records newer than that watermark.

```
Run 1: Process all records up to 2026-03-10T23:59:59Z → watermark = 2026-03-10T23:59:59Z
Run 2: Process records WHERE call_started_at > '2026-03-10T23:59:59Z' → watermark = 2026-03-11T20:00:00Z
Run 3: Process records WHERE call_started_at > '2026-03-11T20:00:00Z' → ...
```

> **In production:** The current LT platform uses a full-refresh Windows Service that rebuilds a flat reporting table every 15 minutes. That's the "before" architecture. The "after" (what you'll build) uses incremental loading — only processing new calls since the last run. This is M09 (Delta Lake MERGE) and M11 (Airflow scheduling).

In [ ]:
# Full refresh strategy: DROP the target table, recreate it from scratch.
# WHY sometimes this is fine: for small tables (<100K rows) or early development,
# full refresh is simpler — no watermark logic, no UPSERT handling.
# The tradeoff: on large tables, this wastes compute re-processing data you already have.

# Full refresh pattern: Drop and recreate a summary table
print("=== Full Refresh: Daily Summary ===")

# DROP TABLE IF EXISTS: safe version of DROP — no error if the table doesn't exist yet.
# WHY IF EXISTS? On first run the table doesn't exist. Plain DROP TABLE would crash.
cursor.execute("DROP TABLE IF EXISTS daily_summary")

# CREATE TABLE ... AS SELECT: create a new table and populate it in one step.
# The column names and types come from the SELECT — no need to define them separately.
cursor.execute("""
    CREATE TABLE daily_summary AS
    SELECT DATE(c.call_started_at, '-5 hours') as call_date_est,  -- Convert UTC → EST
           COUNT(DISTINCT c.call_id) as total_calls,
           SUM(CASE WHEN c.disposition = 'completed' THEN 1 ELSE 0 END) as completed,
           COUNT(DISTINCT o.order_id) as orders,
           ROUND(COALESCE(SUM(o.total), 0), 2) as revenue  -- COALESCE: no orders = $0
    FROM calls c
    LEFT JOIN orders o ON c.call_id = o.call_id
    GROUP BY DATE(c.call_started_at, '-5 hours')  -- Group by EST date, not UTC
""")

print("daily_summary (full refresh):")
for row in cursor.execute("SELECT * FROM daily_summary ORDER BY call_date_est"):
    print(f"  {row[0]}: {row[1]} calls, {row[2]} completed, {row[3]} orders, ${row[4]:,.2f}")

print("\nFull refresh: simple but rebuilds EVERYTHING every time.")
print("On 10M rows, this takes minutes. Incremental takes seconds.")
# You should see 2 date rows: 2026-03-09 and 2026-03-10 (in EST)


In [ ]:
# Incremental loading: only process records that are NEW since the last pipeline run.
# The WATERMARK is the "bookmark" — the timestamp of the last record we already processed.
# On each run: SELECT WHERE timestamp > watermark → process → update watermark to new max.
# WHY: A full refresh of 50M rows takes 20 minutes. Incremental on 50K new rows takes 2 seconds.

# Incremental loading: Watermark + UPSERT pattern

# Step 1: Simulate a watermark — last processed timestamp
# In production, this is stored in a metadata table or pipeline config file, not hardcoded.
last_watermark = "2026-03-10T22:00:00Z"
print(f"Last watermark: {last_watermark}")
print(f"Only processing calls AFTER this timestamp.\n")

# Step 2: Find new records since watermark
# WHERE call_started_at > ?: the ? is a parameterized placeholder (prevents SQL injection)
# Only records after the watermark enter the pipeline for this run.
new_calls = cursor.execute("""
    SELECT call_id, dnis, call_started_at, disposition, channel
    FROM calls
    WHERE call_started_at > ?   -- ← parameterized: SQLite fills in last_watermark safely
    ORDER BY call_started_at
""", (last_watermark,)).fetchall()
# WHY pass as a tuple (last_watermark,)? executemany/execute expect sequence parameters.

print(f"=== New Calls Since Watermark: {len(new_calls)} ===")
for row in new_calls[:5]:
    print(f"  {row[0]} | {row[3]:<12} | {row[2]}")
if len(new_calls) > 5:
    print(f"  ... and {len(new_calls) - 5} more")

# Step 3: Update watermark to the latest timestamp in the new batch
# WHY max()? The records could arrive out of order — max() gives the true latest.
new_watermark = max(row[2] for row in new_calls)
print(f"\nNew watermark: {new_watermark}")
# In production: UPDATE pipeline_metadata SET watermark = new_watermark WHERE pipeline = 'calls'

# Step 4: UPSERT — handle late-arriving or corrected data
# UPSERT = UPDATE if exists, INSERT if not. Handles both new records AND corrections.
print("\n=== UPSERT: Handling Late-Arriving Data ===")
print("Scenario: call_002 was marked 'dropped' but the agent actually completed it.")
print("A correction arrives — we need to UPDATE the existing record.\n")

# Show current record before correction
before = cursor.execute("SELECT call_id, disposition FROM calls WHERE call_id = 'call_002'").fetchone()
print(f"Before: {before[0]} → {before[1]}")

# INSERT OR REPLACE: if call_id (PK) already exists, replace the entire row.
# WHY "OR REPLACE" and not plain UPDATE? Simpler for pipelines — one statement handles both
# new records and corrections without needing IF EXISTS logic.
cursor.execute("""
    INSERT OR REPLACE INTO calls
    VALUES ('call_002', '8005551234', '2485555678', '2026-03-10T13:30:00Z',
            '2026-03-10T13:36:45Z', 'completed', 'neutral', 'live_agent')
    -- ↑ disposition changed from 'dropped' to 'completed' — the correction
""")

after = cursor.execute("SELECT call_id, disposition FROM calls WHERE call_id = 'call_002'").fetchone()
print(f"After:  {after[0]} → {after[1]}")

print("\nINSERT OR REPLACE: if the PK exists, replace the row. If not, insert.")
print("In Delta Lake (M09), you'll use MERGE for this — same concept, more control.")
print("In BigQuery, you'll use MERGE ... WHEN MATCHED THEN UPDATE.")
# BEGINNER NOTE: Late-arriving corrections are common in real pipelines.
# A call marked 'dropped' at 2pm might be corrected to 'completed' after the agent logs it at 3pm.
# Your pipeline must handle this without duplicating the record.


---

## 10. SQL Challenge Lab

**Deliverable:** Complete these 4 challenges. They combine everything from today's module.

Each challenge builds on concepts from multiple sections. Solutions use CTEs, window functions, aggregations, and date handling together.

**How to use this section:** Each challenge shows a "Try it yourself first" prompt. Spend 5 minutes attempting it before revealing the solution. Typing the query yourself — even imperfectly — builds the muscle memory that drills and passive reading don't.

---


### Challenge 1: Top 3 Campaigns by Conversion Rate

**Task:** Find the top 3 campaigns ranked by conversion rate (orders / total calls). Include total calls, orders, revenue, and the conversion rate. Use the call center's `calls`, `orders`, and `sources` tables.

> This was the homework challenge from M02. Now you have CTEs and window functions to solve it cleanly.

---

**Try it yourself first — spend 5 minutes before looking at the solution.**

Think through:
- Which tables do you need? How do they JOIN?
- What's the formula for conversion rate?
- How do you pick only the top 3? (Hint: there are two approaches — `LIMIT 3` vs a window function. Which handles ties correctly?)

```python
# Your attempt here — run this cell, then scroll down for the solution
your_results = cursor.execute("""
    -- Write your query here
    SELECT ...
""").fetchall()
for row in your_results:
    print(row)
```

---
*Solution below.*


In [ ]:
# WHY ROW_NUMBER() here instead of just ORDER BY + LIMIT 3?
# LIMIT 3 doesn't handle ties — if two campaigns have identical conversion rates,
# LIMIT picks one arbitrarily. ROW_NUMBER() lets us see the rank explicitly
# and decide how to handle ties (change to RANK() or DENSE_RANK() if needed).
# This CTE → rank → filter pattern is the standard top-N approach in production SQL.

# Challenge 1: Top 3 campaigns by conversion rate
print("=== Challenge 1: Top 3 Campaigns by Conversion Rate ===")
print(f"{'Rank':>5} {'Campaign':<26} {'Channel':<12} {'Calls':>6} {'Orders':>7} {'Conv%':>7} {'Revenue':>10}")
print("-" * 75)

results = cursor.execute("""
    WITH campaign_metrics AS (
        -- Step 1: Aggregate all metrics per campaign
        SELECT s.campaign_name, s.channel,
               COUNT(DISTINCT c.call_id) as total_calls,
               COUNT(DISTINCT o.order_id) as orders,
               -- Conversion rate: orders placed / total calls (as a percentage)
               ROUND(COUNT(DISTINCT o.order_id) * 100.0 / COUNT(DISTINCT c.call_id), 1) as conv_rate,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue  -- COALESCE: campaigns with no orders = $0
        FROM calls c
        JOIN sources s ON c.dnis = s.dnis   -- INNER JOIN: only calls we can match to a campaign
        LEFT JOIN orders o ON c.call_id = o.call_id  -- LEFT JOIN: keep calls with no order (conv_rate needs them)
        GROUP BY s.campaign_name, s.channel
    ),
    ranked AS (
        -- Step 2: Rank all campaigns by conversion rate (highest first)
        SELECT *,
               ROW_NUMBER() OVER (ORDER BY conv_rate DESC) as rnk
               -- WHY ROW_NUMBER not RANK? For top-N filtering, ROW_NUMBER gives exactly N rows.
               -- RANK would give more than 3 rows if there's a tie at position 3.
        FROM campaign_metrics
    )
    -- Step 3: Filter to top 3 only
    SELECT rnk, campaign_name, channel, total_calls, orders, conv_rate, revenue
    FROM ranked
    WHERE rnk <= 3   -- Keep only rank 1, 2, 3
""").fetchall()

for row in results:
    print(f"{row[0]:>5} {row[1]:<26} {row[2]:<12} {row[3]:>6} {row[4]:>7} {row[5]:>6}% ${row[6]:>9.2f}")

print("\nCTE → ROW_NUMBER → filter. This is the standard top-N pattern.")
# You should see 3 rows. Check: does the highest-ranked campaign make intuitive sense?


### Challenge 2: Daily Revenue with Running Total

**Task:** Show daily revenue (EST) with a running total column. Include call count, order count, and the cumulative revenue across all days.

---

**Try it yourself first — spend 5 minutes before looking at the solution.**

Think through:
- You need daily aggregates first. Which clause groups by day?
- How do you ensure the date is in EST, not UTC?
- Running total is a window function. Which one? What goes in `OVER (...)`?
- Can you do the aggregation and the window function in the same query, or do you need a CTE?

```python
# Your attempt here
your_results = cursor.execute("""
    -- Write your query here
    SELECT ...
""").fetchall()
for row in your_results:
    print(row)
```

---
*Solution below.*


In [ ]:
# WHY two steps (CTE then window function) instead of one query?
# You cannot use a window function directly on a GROUP BY result in the same SELECT.
# The aggregation (GROUP BY) must finish first — that's the CTE.
# Then the window function runs on the aggregated output.
# This two-step pattern (aggregate → window) is extremely common in production SQL.

# Challenge 2: Daily revenue with running total
print("=== Challenge 2: Daily Revenue with Running Total (EST) ===")
print(f"{'Date (EST)':<12} {'Calls':>6} {'Orders':>7} {'Revenue':>10} {'Running Total':>14}")
print("-" * 51)

results = cursor.execute("""
    WITH daily AS (
        -- Step 1: Aggregate to daily level (one row per EST date)
        SELECT DATE(c.call_started_at, '-5 hours') as call_date,  -- UTC → EST conversion
               COUNT(DISTINCT c.call_id) as calls,
               COUNT(DISTINCT o.order_id) as orders,
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue    -- COALESCE: no orders = $0, not NULL
        FROM calls c
        LEFT JOIN orders o ON c.call_id = o.call_id  -- LEFT JOIN: count all calls even those with no order
        GROUP BY DATE(c.call_started_at, '-5 hours')
    )
    -- Step 2: Add running total window function on top of the daily aggregates
    SELECT call_date, calls, orders, revenue,
           -- SUM OVER with ROWS UNBOUNDED PRECEDING = cumulative sum from row 1 to current row
           ROUND(SUM(revenue) OVER (ORDER BY call_date ROWS UNBOUNDED PRECEDING), 2) as running_total
           -- WHY ORDER BY call_date inside OVER? Ensures accumulation goes in chronological order.
    FROM daily
    ORDER BY call_date
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:>6} {row[2]:>7} ${row[3]:>9.2f} ${row[4]:>13.2f}")

print("\nCTE for daily aggregation → window function for running total.")
print("This is exactly how mart_daily_campaign would be built.")
# You should see: running_total for the second day = first day revenue + second day revenue


### Challenge 3: Full Campaign Dashboard

**Task:** Build a single query that produces a complete campaign dashboard: calls, completion rate, orders, conversion rate, revenue, average order value, payment approval rate. Rank by revenue.

---

**Try it yourself first — spend 5 minutes before looking at the solution.**

Think through:
- You need to JOIN 4 tables. Sketch the chain: `calls` → `sources`, `calls` → `orders` → `payments`
- Which JOINs should be LEFT JOIN? (Hint: not all calls have orders, not all orders have payments)
- How do you compute payment approval rate safely when some campaigns have no payments? (Hint: NULLIF)
- Where does RANK() go — inside the main query or in a CTE?

```python
# Your attempt here
your_results = cursor.execute("""
    -- Write your query here
    SELECT ...
""").fetchall()
for row in your_results:
    print(row)
```

---
*Solution below.*


In [ ]:
# WHY CTE + RANK() at the end instead of embedding RANK() in the main query?
# RANK() OVER (ORDER BY revenue DESC) can't be used in the same WHERE/HAVING as the GROUP BY
# that produces revenue — the ranking requires the grouped result to already exist.
# The CTE produces the grouped result; the outer SELECT adds the rank.
# This is the correct two-phase pattern for any ranked aggregation.

# Challenge 3: Full campaign dashboard
print("=== Challenge 3: Campaign Dashboard ===")
print(f"{'Campaign':<24} {'Ch':<4} {'Calls':>5} {'Comp%':>6} {'Ord':>4} {'Conv%':>6} {'Revenue':>9} {'AOV':>7} {'Pay%':>5} {'Rnk':>4}")
print("-" * 80)

results = cursor.execute("""
    WITH dashboard AS (
        SELECT s.campaign_name,
               -- Shorten channel name for display (saves column width)
               CASE s.channel WHEN 'live_agent' THEN 'LA' ELSE 'VA' END as ch,
               -- Total unique calls per campaign (DISTINCT: avoid double-counting from multi-table JOIN)
               COUNT(DISTINCT c.call_id) as calls,
               -- Completion rate: completed calls / total calls (as %)
               ROUND(SUM(CASE WHEN c.disposition = 'completed' THEN 1.0 ELSE 0 END)
                     / COUNT(DISTINCT c.call_id) * 100, 0) as comp_pct,
               -- Orders placed from these calls
               COUNT(DISTINCT o.order_id) as orders,
               -- Conversion rate: orders / total calls (not orders / completed — industry standard)
               ROUND(COUNT(DISTINCT o.order_id) * 100.0
                     / COUNT(DISTINCT c.call_id), 0) as conv_pct,
               -- Total revenue (COALESCE handles campaigns with no orders → $0, not NULL)
               ROUND(COALESCE(SUM(o.total), 0), 2) as revenue,
               -- AOV = Average Order Value: revenue per order (AVG returns NULL if no orders → COALESCE to 0)
               ROUND(COALESCE(AVG(o.total), 0), 2) as aov,
               -- Payment approval rate: approved / total payment attempts
               -- NULLIF(COUNT(p.transaction_id), 0): if no payment attempts, return NULL not divide-by-zero
               ROUND(COALESCE(
                   SUM(CASE WHEN p.status = 'approved' THEN 1.0 ELSE 0 END)
                   / NULLIF(COUNT(p.transaction_id), 0) * 100, 0), 0) as pay_pct
        FROM calls c
        JOIN sources s ON c.dnis = s.dnis       -- INNER JOIN: only calls we can match to a campaign
        LEFT JOIN orders o ON c.call_id = o.call_id       -- LEFT JOIN: keep calls with no order
        LEFT JOIN payments p ON o.order_id = p.order_id  -- LEFT JOIN: keep orders with no payment
        GROUP BY s.campaign_name, s.channel
    )
    SELECT campaign_name, ch, calls, comp_pct, orders, conv_pct,
           revenue, aov, pay_pct,
           RANK() OVER (ORDER BY revenue DESC) as rev_rank  -- Rank AFTER the CTE is computed
    FROM dashboard
    ORDER BY revenue DESC
""").fetchall()

for row in results:
    pay = f"{int(row[8])}%" if row[8] else 'N/A'
    print(f"{row[0]:<24} {row[1]:<4} {row[2]:>5} {int(row[3]):>5}% {row[4]:>4} {int(row[5]):>5}% ${row[6]:>8.2f} ${row[7]:>6.2f} {pay:>5} {row[9]:>4}")

print("\nThis query joins 4 tables, uses CASE WHEN, COALESCE, NULLIF, and RANK.")
print("In production, this would be a dbt model or BigQuery view.")
# You should see 6 rows, ranked 1-6 by revenue. Check that pay% is blank for VA-only campaigns.


### Challenge 4: Find Data Anomalies

**Task:** Write queries to find data quality issues:
1. Calls where channel doesn't match their DNIS source mapping
2. Orders without matching calls (orphaned records)
3. Calls with zero or negative duration
4. Duplicate caller_ani values (same person calling multiple times)

---

**Try it yourself first — spend 5 minutes before looking at the solution.**

Think through:
- For mismatches: you need to compare two columns from two different tables. Which JOIN brings both into scope?
- For orphaned orders: you need orders that have NO matching call. What kind of JOIN + filter finds "no match"?
- For zero duration: how do you calculate duration in SQLite? What's the WHERE clause for "duration <= 0"?
- For repeat callers: which aggregate function counts occurrences? Which clause filters on aggregate values?

```python
# Your attempt here
# Try to write all 4 checks before revealing the solution
```

---
*Solution below.*


In [ ]:
# Data quality checks follow predictable SQL patterns:
# - Mismatches: JOIN two tables, filter WHERE col_a != col_b
# - Orphans: LEFT JOIN, filter WHERE right-side PK IS NULL (the "anti-join" pattern)
# - Impossible values: WHERE with the impossible condition (duration <= 0, amount < 0, etc.)
# - Duplicates: GROUP BY + HAVING COUNT(*) > 1
# These 4 patterns cover 80% of data quality issues you'll encounter in production.

# Challenge 4: Data anomaly detection
print("=== Challenge 4: Data Anomaly Detection ===\n")

# Check 1: Channel mismatches between calls and sources tables
# WHY: calls.channel should always match sources.channel for the same dnis.
# A mismatch means either the call was routed to the wrong platform, or a data entry error.
mismatches = cursor.execute("""
    SELECT c.call_id, c.channel as call_channel, s.channel as source_channel
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis   -- INNER JOIN: we need the source channel to compare
    WHERE c.channel != s.channel        -- Mismatch: the two channel columns disagree
""").fetchall()
print(f"1. Channel mismatches (call vs source): {len(mismatches)}")
if mismatches:
    for row in mismatches:
        print(f"   {row[0]}: call={row[1]}, source={row[2]}")
else:
    print("   None found — data is consistent.")

# Check 2: Orphaned orders (no matching call)
# PATTERN: LEFT JOIN + IS NULL = "anti-join". Returns rows from the LEFT table that have NO match.
# WHY LEFT JOIN? We start from orders and look for calls. If no call matches, calls.call_id = NULL.
orphans = cursor.execute("""
    SELECT o.order_id, o.call_id
    FROM orders o
    LEFT JOIN calls c ON o.call_id = c.call_id  -- LEFT JOIN: keep orders even if no call matches
    WHERE c.call_id IS NULL   -- NULL means no call was found — this is the orphan check
""").fetchall()
print(f"\n2. Orphaned orders (no matching call): {len(orphans)}")
if orphans:
    for row in orphans:
        print(f"   {row[0]} → references missing {row[1]}")
else:
    print("   None found — all orders link to valid calls.")

# Check 3: Zero or negative duration (impossible: a call can't end before it starts)
# WHY check this? Clock drift, system errors, or bad ETL can create impossible timestamps.
bad_duration = cursor.execute("""
    SELECT call_id,
           ROUND((julianday(call_ended_at) - julianday(call_started_at)) * 1440, 1) as dur_min
    FROM calls
    WHERE julianday(call_ended_at) <= julianday(call_started_at)
    -- ↑ ended_at <= started_at means 0 or negative duration — logically impossible
""").fetchall()
print(f"\n3. Zero or negative duration calls: {len(bad_duration)}")
if bad_duration:
    for row in bad_duration:
        print(f"   {row[0]}: {row[1]} min")
else:
    print("   None found — all calls have positive duration.")

# Check 4: Repeat callers (same phone number, multiple calls)
# GROUP BY caller_ani + HAVING COUNT > 1 finds numbers that appear more than once.
# Business interpretation: repeat callers could be callbacks, unresolved issues, or fraud signals.
print(f"\n4. Repeat callers (same phone, multiple calls):")
repeats = cursor.execute("""
    SELECT caller_ani, COUNT(*) as call_count,
           GROUP_CONCAT(DISTINCT disposition) as dispositions  -- List all dispositions for this caller
    FROM calls
    GROUP BY caller_ani
    HAVING COUNT(*) > 1   -- HAVING (not WHERE) because COUNT runs after GROUP BY
    ORDER BY call_count DESC
    LIMIT 5  -- Only show top 5 for readability
""").fetchall()

for row in repeats:
    print(f"   {row[0]}: {row[1]} calls ({row[2]})")

print(f"\n   {len(repeats)} repeat callers found.")
print("   In production, repeat callers could be: callbacks, complaints, or duplicates.")
print("\nThese 4 checks are the foundation of data quality testing (M14).")
# BEGINNER NOTE: In production, these 4 queries become automated data quality tests
# that run every time the pipeline loads new data. Any row returned = a data issue to investigate.


---

## 11. Key Takeaways

1. **HAVING filters groups; WHERE filters rows.** Use HAVING for conditions on aggregates (COUNT, SUM, AVG).
2. **CASE WHEN pivots rows into columns.** Essential for cross-tab reports — no PIVOT command needed.
3. **Subqueries nest queries inside queries.** Useful but can be hard to read — CTEs are usually better.
4. **CTEs break complex queries into named steps.** Read top-to-bottom, reusable, debuggable. dbt is built entirely on CTEs.
5. **Window functions are the DE power tool.** ROW_NUMBER for deduplication, RANK for top-N, LAG/LEAD for period comparison, SUM OVER for running totals.
6. **Always convert to reporting timezone before grouping by date.** The UTC bug is real — evening EST calls appear as the next day in UTC.
7. **COALESCE and NULLIF prevent silent failures.** NULL in a SUM = NULL result. Always handle NULLs explicitly.
8. **EXPLAIN shows how the database executes your query.** SCAN = slow, SEARCH = fast. Indexes turn scans into searches.
9. **Incremental loading processes only what's new.** Watermark pattern tracks the last processed timestamp. Full refresh is simple but doesn't scale.
10. **Data quality checks are SQL queries.** Orphaned records, mismatches, duplicates, impossible values — all found with LEFT JOIN + IS NULL, GROUP BY + HAVING, and CASE WHEN.
11. **These SQL patterns transfer directly to PySpark (M06), BigQuery (M08), and dbt.** The syntax varies slightly but the concepts are identical.

---

## 12. Homework & Preview

### 1. SQL Practice (30 min)

Write these queries using the call center database:

a. **Top 3 hours of the day (EST) by call volume.** Include completion rate per hour.  
b. **Revenue by client with running total.** Use a CTE + window function.  
c. **Find all calls that were completed but had a declined payment.** What's the business impact?

### 2. SQLBolt Lessons 13-18 (30 min)

Complete [SQLBolt](https://sqlbolt.com/) Lessons 13-18 — INSERT, UPDATE, DELETE, CREATE TABLE, ALTER TABLE, DROP TABLE.

### 3. Challenge: The Conversion Funnel by Channel

Build a single CTE query that shows the full conversion funnel (calls → completed → ordered → paid) broken down by channel (live_agent vs va). Which channel has better end-to-end conversion?

### 4. Preview M04: Data Modeling

Skim these concepts — we'll cover them in the next session:
- **Star schema** — fact tables vs dimension tables
- **Slowly Changing Dimensions (SCD Type 2)** — how to track history
- **Partitioning** — splitting large tables for performance

Think about: In our call center data, which table is the fact table? Which are dimensions?

---

**Next session:** M04 — Data Modeling (star schema, SCD Type 2, partitioning)